# ECLIPSE — w/o CSE | Amazon Electronics | seed 3
Fresh **64D, 500 epochs**; only eta changes to **0**, PEEC delta1 and LC .01 remain active. Source is pinned; no validator patch.

- `RUN_MODE='train'`: Add Input (processed MMRec data) **before Save & Run All**, enable GPU and Internet for setup. Training starts only in section 7.
- `RUN_MODE='report_only'`: set native `EXISTING_ARTIFACT_DIR` and optional `EXISTING_STDOUT_LOG`; no source checkout, pip installation, data download, CUDA/model imports or training.
- Unique output per attempt/export; no deletion, overwrite, automatic resume or retry.
- Native evaluator may test terminal and restored selected models. Report only TEST at the VALID NDCG@20-selected checkpoint.
- Historical source/runtime deviations remain visible and require review. No synthetic seed values, SD or p-values for one run.
- Electronics w/o CSE seed3 is an extension of the minimal plan; it does not imply seeds1/2 have been completed.

**2026-10-19 launch fix:** CLI preflight may create an empty artifact directory. Empty real directories without a stdout log are accepted; payload/log/symlink attempts remain protected. No algorithm/config changes.


## 1. Configuration — edit only here


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import sys
import uuid

ARM = 'wo_cse'
SEED = 3
RUN_MODE = 'train'  # or 'report_only'
EXISTING_ARTIFACT_DIR = None  # native directory containing manifest/receipts/checkpoint
EXISTING_STDOUT_LOG = None  # actual stdout .log, needed for Coach.fit timing
NOTEBOOK_SOURCE_PATH = None  # optional exact submitted .ipynb to include
SELECTED_DATASETS = ('electronics',)
GPU_INDEX = 0
INSTALL_MISSING_DEPENDENCIES = True
AUTO_DOWNLOAD_DATA = False  # Optional outside Kaggle only; attach inputs before Save & Run All.
DATASET_HANDLE = 'rainyle/stair-datasets-mmrec'
DATASET_SEARCH_ROOTS = []  # Optional parent folders; default input/cache paths are searched automatically.
STRICT_RUNTIME = True  # Require the seed-1 Python major/minor, Torch and FreeRec versions.
RUNTIME_COMMIT = '5212263e87c5c1cfe6e48c91afa90201a48f5929'
REPO_URL = 'https://github.com/ThanhChuong12/STAIR-Enhanced.git'
REPO_OVERRIDE = None  # Optional existing repository; checked against seed-1 source digests.
DATASET_SOURCE_OVERRIDES = {}  # Example: {'electronics': '/kaggle/input/.../electronics'}
if RUN_MODE not in ('train','report_only'): raise ValueError('Invalid RUN_MODE')
if RUN_MODE=='report_only' and not EXISTING_ARTIFACT_DIR: raise ValueError('Set EXISTING_ARTIFACT_DIR')

if isinstance(SEED, bool) or SEED not in (1, 2, 3):
    raise ValueError('SEED must be 1,2 or3; intended job is Electronics seed3.')
if not SELECTED_DATASETS or len(set(SELECTED_DATASETS)) != len(SELECTED_DATASETS):
    raise ValueError('Select one or more datasets without duplicates.')
if SELECTED_DATASETS != ('electronics',):
    raise ValueError('Unknown dataset key.')
if isinstance(GPU_INDEX, bool) or not isinstance(GPU_INDEX, int) or GPU_INDEX < 0:
    raise ValueError('GPU_INDEX must be a non-negative integer.')

WORK_DIR = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()
SESSION_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8]
RUN_ROOT = WORK_DIR / 'eclipse_runs' / f'eclipse_{ARM}_electronics_seed{SEED}_{SESSION_ID}'
RUN_ROOT.mkdir(parents=True,exist_ok=False)
for name in ('logs', 'artifacts', 'configs', 'figures', 'tables', 'provenance','helpers'):
    (RUN_ROOT / name).mkdir(parents=True, exist_ok=True)
DATA_ROOT = WORK_DIR / 'eclipse_data'
GRAPH_CACHE_ROOT = WORK_DIR / 'eclipse_graph_cache' / RUNTIME_COMMIT[:12] / ARM
GRAPH_CACHE_ROOT.mkdir(parents=True, exist_ok=True)

PRIMARY_METRICS = ('RECALL@10', 'RECALL@20', 'NDCG@10', 'NDCG@20')
DATASET_TITLES = {'baby': 'Amazon Baby', 'sports': 'Amazon Sports', 'electronics': 'Amazon Electronics'}
DATASET_FOLDERS = {k: f'Amazon2014{v}_550_MMRec' for k, v in
                   [('baby', 'Baby'), ('sports', 'Sports'), ('electronics', 'Electronics')]}
print(f'ECLIPSE w/o CSE | seed={SEED} | datasets={SELECTED_DATASETS}')
print(f'Run directory: {RUN_ROOT}')
print('Training starts only in the dataset training cells below.')

STEM=f'eclipse_electronics_{ARM}_seed{SEED}'
ARTIFACT_PATHS={'electronics':Path(EXISTING_ARTIFACT_DIR).expanduser().resolve() if RUN_MODE=='report_only' else RUN_ROOT/'artifacts'/STEM}
LOG_PATHS={'electronics':(Path(EXISTING_STDOUT_LOG).expanduser().resolve() if EXISTING_STDOUT_LOG else None) if RUN_MODE=='report_only' else RUN_ROOT/'logs'/f'{STEM}.log'}


## 2. Full-protocol reference — configurations and fingerprints only, no result injection


In [ ]:
import math, re, time, shutil
def file_sha256(path):
    digest=hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda:handle.read(1024*1024),b''):digest.update(chunk)
    return digest.hexdigest()
def write_json(path,payload):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    temporary=path.with_name(path.name+'.tmp')
    temporary.write_text(json.dumps(payload,indent=2,ensure_ascii=False,allow_nan=False)+'\n',encoding='utf-8')
    temporary.replace(path)
REFERENCE=json.loads('{"commit":"5212263e87c5c1cfe6e48c91afa90201a48f5929","python":"3.13.15","torch":"2.11.0+cu128","freerec":"0.9.7","gpu":"Tesla T4","v8_config":{"embedding_dim":64,"num_layers":3,"gamma":0.4,"lr":0.001,"weight_decay":0.1,"batch_size":4096,"epochs":500,"beta1":0.9,"beta2":0.999,"mfiles":["textual_modality.pkl","visual_modality.pkl"],"num_neighbors":[5,1],"eval_freq":5,"lambda_nlgcl":0.01,"nlgcl_tau":0.2,"nlgcl_G":1,"nlgcl_alpha":0.5,"eta":0.1,"k_cf":5,"c_min":2,"t_shrinkage":5.0,"v8_arm":"WMSG-core","semantic_mode":"weighted","edge_power":2.0,"edge_floor":0.05,"semantic_mix":1.0,"cl_chunk_size":128,"knn_chunk_size":1024,"knn_device":"cpu","cf_block_size":64,"cf_memory_budget_mib":128.0},"icaart_config":{"evidence_mode":"pooled_activity_excess","evidence_activity_bins":"log2","evidence_halves":0,"evidence_seed":null,"evidence_tau":2.0,"evidence_delta":1.0,"evidence_candidate_block":8192,"evidence_placebo_seed":20261016},"features":[{"file":"textual_modality.pkl","sha256":"14a936f18890fbd8b4e6436061b971de878ba79005a2477325d3466e305d635e"},{"file":"visual_modality.pkl","sha256":"13c40105ff6872616e60d0418faa5c4eb4e14e0de1af7d1e71e6820df63dea17"}],"knn":[{"features":"14a936f18890fbd8b4e6436061b971de878ba79005a2477325d3466e305d635e","k":5,"chunk":177,"algorithm":"cosine_fp32_diagonal_minus10_torch_topk_v1","torch":"2.11.0+cu128","device":"cpu","tf32":false,"source":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038","fingerprint":"2459d9b74920fa59d566d62171cbc1434d1c5e98d263d71c6561c2e7b2c9f695","cache_hit":false,"candidate_sha256":"a7be45316c97702a72dcbce8d1595b2dca971af0a132611646ae430522b09d54"},{"features":"13c40105ff6872616e60d0418faa5c4eb4e14e0de1af7d1e71e6820df63dea17","k":1,"chunk":177,"algorithm":"cosine_fp32_diagonal_minus10_torch_topk_v1","torch":"2.11.0+cu128","device":"cpu","tf32":false,"source":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038","fingerprint":"6488fd7aaee647ded9774efd93bfe5c61ee399d4c36a23505569352cb8411b75","cache_hit":false,"candidate_sha256":"536c50370c58b370550c0e947d706b0fc3cadbf32d9f243f2ab0fbc5d0f744eb"}],"contract":{"config":{"embedding_dim":64,"num_layers":3,"gamma":0.4,"num_neighbors":[5,1],"mfiles":["textual_modality.pkl","visual_modality.pkl"],"lr":0.001,"weight_decay":0.1,"beta1":0.9,"beta2":0.999,"batch_size":4096,"seed":1,"eval_freq":5,"ranking":"full","which4best":"NDCG@20","num_workers":4,"nlgcl_G":1,"nlgcl_tau":0.2,"nlgcl_alpha":0.5,"eta":0.1,"k_cf":5,"c_min":2,"t_shrinkage":5.0,"v8_arm":"WMSG-core","semantic_mode":"weighted","edge_power":2.0,"edge_floor":0.05,"semantic_mix":1.0,"alignment":"off","anchor_seed":2718,"anchor_per_stratum":1024,"relation_gate":"off","relation_strength":0.0,"relation_threshold":0.7,"relation_temperature":0.1,"bsc_residual":0.0,"lambda_dirichlet":0.0,"placebo_seed":1,"knn_device":"cpu","knn_chunk_size":1024,"cl_chunk_size":128,"metadata_fingerprint":null},"splits":{"test.txt":"a5f4a1c7eb973d5d6bac5e6f023fcdedf1a66a46805714d233b4eb1fdb468234","train.txt":"495c58aec6c0b23c6780710d58e4718504af26175da8e01855b1f22675062e63","valid.txt":"145ce0ab06bf85260b74ba50829ab5041a2fc5c867d542c81e42da8aefefb51a"},"candidate_hashes":["a7be45316c97702a72dcbce8d1595b2dca971af0a132611646ae430522b09d54","536c50370c58b370550c0e947d706b0fc3cadbf32d9f243f2ab0fbc5d0f744eb"],"sources":{"stair5_v8.py":"dea612a122c82f7cc6c4c1ba069510fa1242fa3819f00b6cbfe0d37b9ed9c502","stair5_v8_alignment.py":"8d6bb9eb25dfe2426d2a358ec82268d27c9b4913eef0472b6fda83fbd231607d","stair5_v8_graph.py":"2a7c931740ceeac177ce3fc6dc2cdae479eb75673a8ef1b115122ba4a0826ae7","stair5_v4_graph.py":"eb5d2b2355871711931d597bbe2d01f520b174852430e922be268826c7252241","stair5_v4.py":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038","stair5_v4_objectives.py":"1afd078a22aad8635778cfa61f05628beaecd180cd2c6c17ed56f72a6a94182e","stair5_v4_utils.py":"39930cb54af3f5cdb65983f05b3b03cabe502673b9d70889b0ac7c1d22274b1a","freerec_compat.py":"de46f3dd9bc11bb69acd5f151d175968e73d980be37ad36d5afc636b0b9c5e90","stair5_v8_smoother.py":"c4f1ebab6008ffa5f7720007a17854040f0a7211fbd6087fc9dedd26d10d7663","AdamW.py":"43108b6264cd5b1fa723d0a731faff0269d05fb91bcf525445c9bd0406443715","main_stair5_v8.py":"036b45594a8c0b17e32c5401c5b84a93e85992fadae6303aa0db9eeaea8331b8","stair_icaart.py":"77b862894f9aa91eba8fa96ad54a38eb6e29a25d9e134b7718a58f4bb5851014","stair_icaart_evidence.py":"11487e0fef87077cde7863b58847bcf22adeebef6e5378ae32f71f366d88c41a","stair_icaart_graph.py":"33bd582c8cd3b298eed1ce47ad99837ec9f0cd220778d684ae3d96161a455aad","stair_icaart_config.py":"fe5bcaa711ba64fd40d431db27ae6278e7c937ce05f57786b6f7262668061a51","main_stair_icaart.py":"b95eb5423ac8947da290bd13562e0aeb433adb8742ca0f0246139b4880aeec3f","stair_icaart_reporting.py":"6902360e9ac92dd511ea87e988ad622cae51886eb0ac183b170ce7df1b5065cc"},"data_fingerprint":"84680e889deeecb904f4475620eb0d638da332a95202c193c73c68924b0f0a82","graph_fingerprint":"43b9d0d255d7c7e72322fbeb834b0bc08206fcb236680db373aa30b6b6040242"},"provenance":"logs/GD6/STAIR-PEEC/electronics/stair_icaart/electronics_STAIR_ICAART_pooled_activity_excess_d1.0_seed1","evidence_invariants":{"raw_semantic_fingerprint":"dbaae7d2b6e7efae9e8a70d1f9141e2cbd5b365cb8a0bb455175ccf5b5e6e28c","assignment_hash":"6e8bac97c4b65448a184f088c4e7cfb0f3ec534edfd8e67ca4e9e954fac93584","evidence_score_hash":"9f4bcf6e5a5d517e4ed9518c9a7e99eabbcd9fbf442195fdf2933e27ed4364b4","counts_shape":[271356],"null_expectation_shape":[271356],"counts_hash":"acd4ee8f925d91ac5552260927d1a505ed2514bbf0af47445316eabf3e3a0607","null_expectation_hash":"042d89622e4c63558e14dfb888e81a7aae75844689f0f7bc979fcc731e440582","train_presence_hash":"c52cc4f40aa6657e6738a9f96a9414a908074a1355df1d189f7f27006dbe257e","num_raw_train_interactions":1254441,"num_deduplicated_train_interactions":1254441,"num_users":192403,"num_items":63001,"group_sizes":[0,59094,88084,34713,8528,1608,310,60,6]}}')
SEED1_REFERENCE={'electronics':REFERENCE}
write_json(RUN_ROOT/'provenance'/'protocol_reference.json',REFERENCE)
write_json(RUN_ROOT/'run_manifest.json',{'arm_id':ARM,'dataset':'electronics','seed':SEED,'session_id':SESSION_ID,'mode':RUN_MODE,'status':'PREPARED','purpose':'component_ablation','expected_algorithm_diff':{'eta':[0.1,0.0]}})

NOTEBOOK_SOURCE_TEMPLATE=json.loads('{"cells":[{"cell_type":"markdown","id":"d2178299","metadata":{},"source":["# ECLIPSE — w/o CSE | Amazon Electronics | seed 3\\n","Fresh **64D, 500 epochs**; only eta changes to **0**, PEEC delta1 and LC .01 remain active. Source is pinned; no validator patch.\\n","\\n","- `RUN_MODE=\'train\'`: Add Input (processed MMRec data) **before Save & Run All**, enable GPU and Internet for setup. Training starts only in section 7.\\n","- `RUN_MODE=\'report_only\'`: set native `EXISTING_ARTIFACT_DIR` and optional `EXISTING_STDOUT_LOG`; no source checkout, pip installation, data download, CUDA/model imports or training.\\n","- Unique output per attempt/export; no deletion, overwrite, automatic resume or retry.\\n","- Native evaluator may test terminal and restored selected models. Report only TEST at the VALID NDCG@20-selected checkpoint.\\n","- Historical source/runtime deviations remain visible and require review. No synthetic seed values, SD or p-values for one run.\\n","- Electronics w/o CSE seed3 is an extension of the minimal plan; it does not imply seeds1/2 have been completed.\\n","\\n","**2026-10-19 launch fix:** CLI preflight may create an empty artifact directory. Empty real directories without a stdout log are accepted; payload/log/symlink attempts remain protected. No algorithm/config changes.\\n"]},{"cell_type":"markdown","id":"632a4e32","metadata":{},"source":["## 1. Configuration — edit only here\\n"]},{"cell_type":"code","id":"c604eca0","metadata":{},"source":["from pathlib import Path\\n","from datetime import datetime, timezone\\n","import hashlib\\n","import json\\n","import os\\n","import sys\\n","import uuid\\n","\\n","ARM = \'wo_cse\'\\n","SEED = 3\\n","RUN_MODE = \'train\'  # or \'report_only\'\\n","EXISTING_ARTIFACT_DIR = None  # native directory containing manifest/receipts/checkpoint\\n","EXISTING_STDOUT_LOG = None  # actual stdout .log, needed for Coach.fit timing\\n","NOTEBOOK_SOURCE_PATH = None  # optional exact submitted .ipynb to include\\n","SELECTED_DATASETS = (\'electronics\',)\\n","GPU_INDEX = 0\\n","INSTALL_MISSING_DEPENDENCIES = True\\n","AUTO_DOWNLOAD_DATA = False  # Optional outside Kaggle only; attach inputs before Save & Run All.\\n","DATASET_HANDLE = \'rainyle/stair-datasets-mmrec\'\\n","DATASET_SEARCH_ROOTS = []  # Optional parent folders; default input/cache paths are searched automatically.\\n","STRICT_RUNTIME = True  # Require the seed-1 Python major/minor, Torch and FreeRec versions.\\n","RUNTIME_COMMIT = \'5212263e87c5c1cfe6e48c91afa90201a48f5929\'\\n","REPO_URL = \'https://github.com/ThanhChuong12/STAIR-Enhanced.git\'\\n","REPO_OVERRIDE = None  # Optional existing repository; checked against seed-1 source digests.\\n","DATASET_SOURCE_OVERRIDES = {}  # Example: {\'electronics\': \'/kaggle/input/.../electronics\'}\\n","if RUN_MODE not in (\'train\',\'report_only\'): raise ValueError(\'Invalid RUN_MODE\')\\n","if RUN_MODE==\'report_only\' and not EXISTING_ARTIFACT_DIR: raise ValueError(\'Set EXISTING_ARTIFACT_DIR\')\\n","\\n","if isinstance(SEED, bool) or SEED not in (1, 2, 3):\\n","    raise ValueError(\'SEED must be 1,2 or3; intended job is Electronics seed3.\')\\n","if not SELECTED_DATASETS or len(set(SELECTED_DATASETS)) != len(SELECTED_DATASETS):\\n","    raise ValueError(\'Select one or more datasets without duplicates.\')\\n","if SELECTED_DATASETS != (\'electronics\',):\\n","    raise ValueError(\'Unknown dataset key.\')\\n","if isinstance(GPU_INDEX, bool) or not isinstance(GPU_INDEX, int) or GPU_INDEX < 0:\\n","    raise ValueError(\'GPU_INDEX must be a non-negative integer.\')\\n","\\n","WORK_DIR = Path(\'/kaggle/working\') if Path(\'/kaggle/working\').is_dir() else Path.cwd()\\n","SESSION_ID = datetime.now(timezone.utc).strftime(\'%Y%m%dT%H%M%SZ\') + \'_\' + uuid.uuid4().hex[:8]\\n","RUN_ROOT = WORK_DIR / \'eclipse_runs\' / f\'eclipse_{ARM}_electronics_seed{SEED}_{SESSION_ID}\'\\n","RUN_ROOT.mkdir(parents=True,exist_ok=False)\\n","for name in (\'logs\', \'artifacts\', \'configs\', \'figures\', \'tables\', \'provenance\',\'helpers\'):\\n","    (RUN_ROOT / name).mkdir(parents=True, exist_ok=True)\\n","DATA_ROOT = WORK_DIR / \'eclipse_data\'\\n","GRAPH_CACHE_ROOT = WORK_DIR / \'eclipse_graph_cache\' / RUNTIME_COMMIT[:12] / ARM\\n","GRAPH_CACHE_ROOT.mkdir(parents=True, exist_ok=True)\\n","\\n","PRIMARY_METRICS = (\'RECALL@10\', \'RECALL@20\', \'NDCG@10\', \'NDCG@20\')\\n","DATASET_TITLES = {\'baby\': \'Amazon Baby\', \'sports\': \'Amazon Sports\', \'electronics\': \'Amazon Electronics\'}\\n","DATASET_FOLDERS = {k: f\'Amazon2014{v}_550_MMRec\' for k, v in\\n","                   [(\'baby\', \'Baby\'), (\'sports\', \'Sports\'), (\'electronics\', \'Electronics\')]}\\n","print(f\'ECLIPSE w/o CSE | seed={SEED} | datasets={SELECTED_DATASETS}\')\\n","print(f\'Run directory: {RUN_ROOT}\')\\n","print(\'Training starts only in the dataset training cells below.\')\\n","\\n","STEM=f\'eclipse_electronics_{ARM}_seed{SEED}\'\\n","ARTIFACT_PATHS={\'electronics\':Path(EXISTING_ARTIFACT_DIR).expanduser().resolve() if RUN_MODE==\'report_only\' else RUN_ROOT/\'artifacts\'/STEM}\\n","LOG_PATHS={\'electronics\':(Path(EXISTING_STDOUT_LOG).expanduser().resolve() if EXISTING_STDOUT_LOG else None) if RUN_MODE==\'report_only\' else RUN_ROOT/\'logs\'/f\'{STEM}.log\'}\\n"],"execution_count":null,"outputs":[]},{"cell_type":"markdown","id":"f4006492","metadata":{},"source":["## 2. Full-protocol reference — configurations and fingerprints only, no result injection\\n"]},{"cell_type":"code","id":"db9ae641","metadata":{},"source":["import math, re, time, shutil\\n","def file_sha256(path):\\n","    digest=hashlib.sha256()\\n","    with Path(path).open(\'rb\') as handle:\\n","        for chunk in iter(lambda:handle.read(1024*1024),b\'\'):digest.update(chunk)\\n","    return digest.hexdigest()\\n","def write_json(path,payload):\\n","    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\\n","    temporary=path.with_name(path.name+\'.tmp\')\\n","    temporary.write_text(json.dumps(payload,indent=2,ensure_ascii=False,allow_nan=False)+\'\\\\n\',encoding=\'utf-8\')\\n","    temporary.replace(path)\\n","REFERENCE=json.loads(\'{\\"commit\\":\\"5212263e87c5c1cfe6e48c91afa90201a48f5929\\",\\"python\\":\\"3.13.15\\",\\"torch\\":\\"2.11.0+cu128\\",\\"freerec\\":\\"0.9.7\\",\\"gpu\\":\\"Tesla T4\\",\\"v8_config\\":{\\"embedding_dim\\":64,\\"num_layers\\":3,\\"gamma\\":0.4,\\"lr\\":0.001,\\"weight_decay\\":0.1,\\"batch_size\\":4096,\\"epochs\\":500,\\"beta1\\":0.9,\\"beta2\\":0.999,\\"mfiles\\":[\\"textual_modality.pkl\\",\\"visual_modality.pkl\\"],\\"num_neighbors\\":[5,1],\\"eval_freq\\":5,\\"lambda_nlgcl\\":0.01,\\"nlgcl_tau\\":0.2,\\"nlgcl_G\\":1,\\"nlgcl_alpha\\":0.5,\\"eta\\":0.1,\\"k_cf\\":5,\\"c_min\\":2,\\"t_shrinkage\\":5.0,\\"v8_arm\\":\\"WMSG-core\\",\\"semantic_mode\\":\\"weighted\\",\\"edge_power\\":2.0,\\"edge_floor\\":0.05,\\"semantic_mix\\":1.0,\\"cl_chunk_size\\":128,\\"knn_chunk_size\\":1024,\\"knn_device\\":\\"cpu\\",\\"cf_block_size\\":64,\\"cf_memory_budget_mib\\":128.0},\\"icaart_config\\":{\\"evidence_mode\\":\\"pooled_activity_excess\\",\\"evidence_activity_bins\\":\\"log2\\",\\"evidence_halves\\":0,\\"evidence_seed\\":null,\\"evidence_tau\\":2.0,\\"evidence_delta\\":1.0,\\"evidence_candidate_block\\":8192,\\"evidence_placebo_seed\\":20261016},\\"features\\":[{\\"file\\":\\"textual_modality.pkl\\",\\"sha256\\":\\"14a936f18890fbd8b4e6436061b971de878ba79005a2477325d3466e305d635e\\"},{\\"file\\":\\"visual_modality.pkl\\",\\"sha256\\":\\"13c40105ff6872616e60d0418faa5c4eb4e14e0de1af7d1e71e6820df63dea17\\"}],\\"knn\\":[{\\"features\\":\\"14a936f18890fbd8b4e6436061b971de878ba79005a2477325d3466e305d635e\\",\\"k\\":5,\\"chunk\\":177,\\"algorithm\\":\\"cosine_fp32_diagonal_minus10_torch_topk_v1\\",\\"torch\\":\\"2.11.0+cu128\\",\\"device\\":\\"cpu\\",\\"tf32\\":false,\\"source\\":\\"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038\\",\\"fingerprint\\":\\"2459d9b74920fa59d566d62171cbc1434d1c5e98d263d71c6561c2e7b2c9f695\\",\\"cache_hit\\":false,\\"candidate_sha256\\":\\"a7be45316c97702a72dcbce8d1595b2dca971af0a132611646ae430522b09d54\\"},{\\"features\\":\\"13c40105ff6872616e60d0418faa5c4eb4e14e0de1af7d1e71e6820df63dea17\\",\\"k\\":1,\\"chunk\\":177,\\"algorithm\\":\\"cosine_fp32_diagonal_minus10_torch_topk_v1\\",\\"torch\\":\\"2.11.0+cu128\\",\\"device\\":\\"cpu\\",\\"tf32\\":false,\\"source\\":\\"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038\\",\\"fingerprint\\":\\"6488fd7aaee647ded9774efd93bfe5c61ee399d4c36a23505569352cb8411b75\\",\\"cache_hit\\":false,\\"candidate_sha256\\":\\"536c50370c58b370550c0e947d706b0fc3cadbf32d9f243f2ab0fbc5d0f744eb\\"}],\\"contract\\":{\\"config\\":{\\"embedding_dim\\":64,\\"num_layers\\":3,\\"gamma\\":0.4,\\"num_neighbors\\":[5,1],\\"mfiles\\":[\\"textual_modality.pkl\\",\\"visual_modality.pkl\\"],\\"lr\\":0.001,\\"weight_decay\\":0.1,\\"beta1\\":0.9,\\"beta2\\":0.999,\\"batch_size\\":4096,\\"seed\\":1,\\"eval_freq\\":5,\\"ranking\\":\\"full\\",\\"which4best\\":\\"NDCG@20\\",\\"num_workers\\":4,\\"nlgcl_G\\":1,\\"nlgcl_tau\\":0.2,\\"nlgcl_alpha\\":0.5,\\"eta\\":0.1,\\"k_cf\\":5,\\"c_min\\":2,\\"t_shrinkage\\":5.0,\\"v8_arm\\":\\"WMSG-core\\",\\"semantic_mode\\":\\"weighted\\",\\"edge_power\\":2.0,\\"edge_floor\\":0.05,\\"semantic_mix\\":1.0,\\"alignment\\":\\"off\\",\\"anchor_seed\\":2718,\\"anchor_per_stratum\\":1024,\\"relation_gate\\":\\"off\\",\\"relation_strength\\":0.0,\\"relation_threshold\\":0.7,\\"relation_temperature\\":0.1,\\"bsc_residual\\":0.0,\\"lambda_dirichlet\\":0.0,\\"placebo_seed\\":1,\\"knn_device\\":\\"cpu\\",\\"knn_chunk_size\\":1024,\\"cl_chunk_size\\":128,\\"metadata_fingerprint\\":null},\\"splits\\":{\\"test.txt\\":\\"a5f4a1c7eb973d5d6bac5e6f023fcdedf1a66a46805714d233b4eb1fdb468234\\",\\"train.txt\\":\\"495c58aec6c0b23c6780710d58e4718504af26175da8e01855b1f22675062e63\\",\\"valid.txt\\":\\"145ce0ab06bf85260b74ba50829ab5041a2fc5c867d542c81e42da8aefefb51a\\"},\\"candidate_hashes\\":[\\"a7be45316c97702a72dcbce8d1595b2dca971af0a132611646ae430522b09d54\\",\\"536c50370c58b370550c0e947d706b0fc3cadbf32d9f243f2ab0fbc5d0f744eb\\"],\\"sources\\":{\\"stair5_v8.py\\":\\"dea612a122c82f7cc6c4c1ba069510fa1242fa3819f00b6cbfe0d37b9ed9c502\\",\\"stair5_v8_alignment.py\\":\\"8d6bb9eb25dfe2426d2a358ec82268d27c9b4913eef0472b6fda83fbd231607d\\",\\"stair5_v8_graph.py\\":\\"2a7c931740ceeac177ce3fc6dc2cdae479eb75673a8ef1b115122ba4a0826ae7\\",\\"stair5_v4_graph.py\\":\\"eb5d2b2355871711931d597bbe2d01f520b174852430e922be268826c7252241\\",\\"stair5_v4.py\\":\\"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038\\",\\"stair5_v4_objectives.py\\":\\"1afd078a22aad8635778cfa61f05628beaecd180cd2c6c17ed56f72a6a94182e\\",\\"stair5_v4_utils.py\\":\\"39930cb54af3f5cdb65983f05b3b03cabe502673b9d70889b0ac7c1d22274b1a\\",\\"freerec_compat.py\\":\\"de46f3dd9bc11bb69acd5f151d175968e73d980be37ad36d5afc636b0b9c5e90\\",\\"stair5_v8_smoother.py\\":\\"c4f1ebab6008ffa5f7720007a17854040f0a7211fbd6087fc9dedd26d10d7663\\",\\"AdamW.py\\":\\"43108b6264cd5b1fa723d0a731faff0269d05fb91bcf525445c9bd0406443715\\",\\"main_stair5_v8.py\\":\\"036b45594a8c0b17e32c5401c5b84a93e85992fadae6303aa0db9eeaea8331b8\\",\\"stair_icaart.py\\":\\"77b862894f9aa91eba8fa96ad54a38eb6e29a25d9e134b7718a58f4bb5851014\\",\\"stair_icaart_evidence.py\\":\\"11487e0fef87077cde7863b58847bcf22adeebef6e5378ae32f71f366d88c41a\\",\\"stair_icaart_graph.py\\":\\"33bd582c8cd3b298eed1ce47ad99837ec9f0cd220778d684ae3d96161a455aad\\",\\"stair_icaart_config.py\\":\\"fe5bcaa711ba64fd40d431db27ae6278e7c937ce05f57786b6f7262668061a51\\",\\"main_stair_icaart.py\\":\\"b95eb5423ac8947da290bd13562e0aeb433adb8742ca0f0246139b4880aeec3f\\",\\"stair_icaart_reporting.py\\":\\"6902360e9ac92dd511ea87e988ad622cae51886eb0ac183b170ce7df1b5065cc\\"},\\"data_fingerprint\\":\\"84680e889deeecb904f4475620eb0d638da332a95202c193c73c68924b0f0a82\\",\\"graph_fingerprint\\":\\"43b9d0d255d7c7e72322fbeb834b0bc08206fcb236680db373aa30b6b6040242\\"},\\"provenance\\":\\"logs/GD6/STAIR-PEEC/electronics/stair_icaart/electronics_STAIR_ICAART_pooled_activity_excess_d1.0_seed1\\",\\"evidence_invariants\\":{\\"raw_semantic_fingerprint\\":\\"dbaae7d2b6e7efae9e8a70d1f9141e2cbd5b365cb8a0bb455175ccf5b5e6e28c\\",\\"assignment_hash\\":\\"6e8bac97c4b65448a184f088c4e7cfb0f3ec534edfd8e67ca4e9e954fac93584\\",\\"evidence_score_hash\\":\\"9f4bcf6e5a5d517e4ed9518c9a7e99eabbcd9fbf442195fdf2933e27ed4364b4\\",\\"counts_shape\\":[271356],\\"null_expectation_shape\\":[271356],\\"counts_hash\\":\\"acd4ee8f925d91ac5552260927d1a505ed2514bbf0af47445316eabf3e3a0607\\",\\"null_expectation_hash\\":\\"042d89622e4c63558e14dfb888e81a7aae75844689f0f7bc979fcc731e440582\\",\\"train_presence_hash\\":\\"c52cc4f40aa6657e6738a9f96a9414a908074a1355df1d189f7f27006dbe257e\\",\\"num_raw_train_interactions\\":1254441,\\"num_deduplicated_train_interactions\\":1254441,\\"num_users\\":192403,\\"num_items\\":63001,\\"group_sizes\\":[0,59094,88084,34713,8528,1608,310,60,6]}}\')\\n","SEED1_REFERENCE={\'electronics\':REFERENCE}\\n","write_json(RUN_ROOT/\'provenance\'/\'protocol_reference.json\',REFERENCE)\\n","write_json(RUN_ROOT/\'run_manifest.json\',{\'arm_id\':ARM,\'dataset\':\'electronics\',\'seed\':SEED,\'session_id\':SESSION_ID,\'mode\':RUN_MODE,\'status\':\'PREPARED\',\'purpose\':\'component_ablation\',\'expected_algorithm_diff\':{\'eta\':[0.1,0.0]}})\\n"],"execution_count":null,"outputs":[]},{"cell_type":"markdown","id":"c4bbcc8b","metadata":{},"source":["## 3. Pinned source and environment — train mode only; no in-place patches or CPU fallback\\n"]},{"cell_type":"code","id":"0d764683","metadata":{},"source":["if RUN_MODE == \'train\':\\n","    import importlib.metadata\\n","    import importlib.util\\n","    import subprocess\\n","\\n","    if REPO_OVERRIDE is not None:\\n","        REPO_ROOT=Path(REPO_OVERRIDE).expanduser().resolve()\\n","    else:\\n","        REPO_ROOT=WORK_DIR/\'eclipse_sources\'/RUNTIME_COMMIT\\n","        if not REPO_ROOT.exists():\\n","            REPO_ROOT.mkdir(parents=True,exist_ok=False)\\n","            subprocess.run([\'git\',\'init\',str(REPO_ROOT)],check=True)\\n","            subprocess.run([\'git\',\'-C\',str(REPO_ROOT),\'remote\',\'add\',\'origin\',REPO_URL],check=True)\\n","            subprocess.run([\'git\',\'-C\',str(REPO_ROOT),\'fetch\',\'--depth\',\'1\',\'origin\',RUNTIME_COMMIT],check=True)\\n","            subprocess.run([\'git\',\'-C\',str(REPO_ROOT),\'checkout\',\'--detach\',RUNTIME_COMMIT],check=True)\\n","    actual=subprocess.run([\'git\',\'-C\',str(REPO_ROOT),\'rev-parse\',\'HEAD\'],text=True,capture_output=True,check=True).stdout.strip()\\n","    if actual!=RUNTIME_COMMIT:raise RuntimeError(\'Use the pinned checkout; existing source is never reset.\')\\n","    for args,name in (([\'status\',\'--porcelain\'],\'git_status.txt\'),([\'diff\',\'HEAD\',\'--\'],\'source_diff.patch\'),([\'remote\',\'-v\'],\'git_remotes.txt\')):\\n","        result=subprocess.run([\'git\',\'-C\',str(REPO_ROOT),*args],text=True,encoding=\'utf-8\',capture_output=True,check=True)\\n","        (RUN_ROOT/\'provenance\'/name).write_text(result.stdout,encoding=\'utf-8\')\\n","    if not (REPO_ROOT / \'main_stair_icaart.py\').is_file():\\n","        raise FileNotFoundError(REPO_ROOT / \'main_stair_icaart.py\')\\n","\\n","    ENV = os.environ.copy()\\n","    ENV[\'PYTHONPATH\'] = os.pathsep.join(filter(None, [str(REPO_ROOT), ENV.get(\'PYTHONPATH\', \'\')]))\\n","    ENV[\'PYTHONUTF8\'] = \'1\'\\n","    ENV[\'PYTHONIOENCODING\'] = \'utf-8\'\\n","    ENV[\'PYTHONHASHSEED\'] = str(SEED)\\n","    ENV[\'CUDA_VISIBLE_DEVICES\'] = str(GPU_INDEX)\\n","    ENV.setdefault(\'CUBLAS_WORKSPACE_CONFIG\', \':4096:8\')\\n","\\n","    # Do not import Torch into the notebook kernel: all CUDA work belongs to the training subprocess.\\n","    if INSTALL_MISSING_DEPENDENCIES:\\n","        packages = {\\n","            \'freerec\': (\'freerec==0.9.7\', True),\\n","            \'torch_geometric\': (\'torch-geometric==2.8.1\', True),\\n","            \'numpy\': (\'numpy==2.1.3\', True), \'scipy\': (\'scipy==1.16.3\', True),\\n","            \'yaml\': (\'PyYAML==6.0.3\', True), \'pandas\': (\'pandas==2.3.3\', True),\\n","            \'matplotlib\': (\'matplotlib==3.10.0\', True), \'prettytable\': (\'prettytable==3.18.0\', True),\\n","            \'sklearn\': (\'scikit-learn==1.6.1\', True), \'psutil\': (\'psutil==5.9.5\', True),\\n","        }\\n","        for module, (requirement, no_deps) in packages.items():\\n","            missing = importlib.util.find_spec(module) is None\\n","            if module == \'freerec\' and not missing:\\n","                missing = importlib.metadata.version(\'freerec\') != \'0.9.7\'\\n","            if missing:\\n","                command = [sys.executable, \'-m\', \'pip\', \'install\', \'--quiet\']\\n","                if no_deps:\\n","                    command.append(\'--no-deps\')\\n","                subprocess.run(command + [requirement], check=True)\\n","\\n","    expected_sources = SEED1_REFERENCE[\'electronics\'][\'contract\'][\'sources\']\\n","    source_report = {}\\n","    for name, expected in expected_sources.items():\\n","        choices = [REPO_ROOT / name, REPO_ROOT / \'models\' / name, REPO_ROOT / \'optimizers\' / name]\\n","        path = next((p for p in choices if p.is_file()), None)\\n","        if path is None:\\n","            raise FileNotFoundError(f\'Missing seed-1 source: {name}\')\\n","        payload = path.read_bytes()\\n","        actual = hashlib.sha256(payload).hexdigest()\\n","        lf_digest = hashlib.sha256(payload.replace(b\'\\\\r\\\\n\', b\'\\\\n\')).hexdigest()\\n","        source_report[name] = {\'path\': str(path.relative_to(REPO_ROOT)), \'sha256\': actual,\\n","                               \'line_endings_only\': actual != expected and lf_digest == expected}\\n","        if expected not in (actual, lf_digest):\\n","            raise RuntimeError(f\'Algorithm source differs from seed 1: {path}. Use the pinned runtime commit.\')\\n","    write_json(RUN_ROOT / \'provenance\' / \'source_check.json\', source_report)\\n","\\n","    probe = r\\"\\"\\"\\n","    import json, platform, torch\\n","    import models.freerec_compat\\n","    import freerec\\n","    from main_stair_icaart import build_config\\n","    from stair_icaart_reporting import validate_selected_receipt\\n","    print(\'ECLIPSE_ENV=\' + json.dumps({\'python\': platform.python_version(), \'torch\': str(torch.__version__),\\n","     \'freerec\': str(freerec.__version__), \'torch_geometric\': __import__(\'importlib.metadata\',fromlist=[\'version\']).version(\'torch-geometric\'), \'cuda\': torch.cuda.is_available(),\\n","     \'gpu\': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}))\\n","    \\"\\"\\"\\n","    probe = __import__(\'textwrap\').dedent(probe)\\n","    checked = subprocess.run([sys.executable, \'-c\', probe], cwd=REPO_ROOT, env=ENV,\\n","                             text=True, encoding=\'utf-8\', capture_output=True)\\n","    if checked.stdout:\\n","        print(checked.stdout)\\n","    if checked.stderr:\\n","        print(checked.stderr)\\n","    checked.check_returncode()\\n","    line = next(line for line in checked.stdout.splitlines() if line.startswith(\'ECLIPSE_ENV=\'))\\n","    RUNTIME_ENV = json.loads(line.split(\'=\', 1)[1])\\n","    if not RUNTIME_ENV[\'cuda\']:\\n","        raise RuntimeError(\'Enable a Kaggle GPU accelerator before training.\')\\n","    reference = SEED1_REFERENCE[\'electronics\']\\n","    deviations = {key: {\'reference\': reference[key], \'current\': RUNTIME_ENV[key]}\\n","                  for key in (\'torch\', \'freerec\',\'gpu\') if RUNTIME_ENV[key] != reference[key]}\\n","    if RUNTIME_ENV[\'python\'].split(\'.\')[:2] != reference[\'python\'].split(\'.\')[:2]:\\n","        deviations[\'python\'] = {\'reference\': reference[\'python\'], \'current\': RUNTIME_ENV[\'python\']}\\n","    if RUNTIME_ENV.get(\'torch_geometric\') != \'2.8.1\':\\n","        deviations[\'torch_geometric\'] = {\'reference\':\'2.8.1\',\'current\':RUNTIME_ENV.get(\'torch_geometric\')}\\n","    write_json(RUN_ROOT / \'provenance\' / \'environment.json\', {\'runtime\': RUNTIME_ENV, \'deviations\': deviations,\\n","               \'strict_runtime\': STRICT_RUNTIME, \'gpu_index\': GPU_INDEX})\\n","    if deviations and STRICT_RUNTIME:\\n","        raise RuntimeError(f\'Runtime differs from seed 1: {deviations}. Restore the seed-1 environment; \'\\n","                           \'setting STRICT_RUNTIME=False explicitly records a protocol deviation.\')\\n","    if deviations:\\n","        print(\'PROTOCOL DEVIATION:\', deviations)\\n","    if RUNTIME_ENV[\'gpu\'] != reference[\'gpu\']:\\n","        print(\'HARDWARE DEVIATION:\', RUNTIME_ENV[\'gpu\'], \'vs\', reference[\'gpu\'],\\n","              \'- runtime measurements are not a matched-hardware comparison.\')\\n","    freeze = subprocess.run([sys.executable, \'-m\', \'pip\', \'freeze\'], text=True, capture_output=True, check=True)\\n","    (RUN_ROOT / \'provenance\' / \'pip_freeze.txt\').write_text(freeze.stdout, encoding=\'utf-8\')\\n","    commit = subprocess.run([\'git\', \'-C\', str(REPO_ROOT), \'rev-parse\', \'HEAD\'], text=True, capture_output=True)\\n","    ACTUAL_COMMIT = commit.stdout.strip() if commit.returncode == 0 else \'unavailable\'\\n","    print(\'Runtime ready:\', REPO_ROOT, \'| commit:\', ACTUAL_COMMIT)\\n","    print(\'Import/environment checks only; no model test or training has been executed.\')\\n","\\n","    driver=subprocess.run([\'nvidia-smi\',\'--query-gpu=name,driver_version,memory.total\',\'--format=csv\'],text=True,capture_output=True)\\n","    (RUN_ROOT/\'provenance\'/\'nvidia_smi.txt\').write_text(driver.stdout+driver.stderr,encoding=\'utf-8\')\\n","\\n","else:\\n","    print(\'report_only: skipped training setup; no model/network/GPU invocation.\')\\n"],"execution_count":null,"outputs":[]},{"cell_type":"markdown","id":"ba79db6b","metadata":{},"source":["## 4. Input discovery and split/feature hashes — attach data before Kaggle batch; report-only skips data\\n"]},{"cell_type":"code","id":"2c313da3","metadata":{},"source":["if RUN_MODE == \'train\':\\n","    import shutil\\n","\\n","    REQUIRED_FILES = (\'train.txt\', \'valid.txt\', \'test.txt\', \'textual_modality.pkl\', \'visual_modality.pkl\')\\n","\\n","\\n","    def dataset_search_roots(extra_roots=()):\\n","        # Include writable bridges and caches created by earlier interactive sessions.\\n","        return [REPO_ROOT / \'data\', DATA_ROOT, WORK_DIR / \'data\',\\n","                Path(\'/kaggle/data\'), Path(\'/kaggle/input\'),\\n","                Path.home() / \'.cache\' / \'kagglehub\' / \'datasets\',\\n","                *map(Path, DATASET_SEARCH_ROOTS), *map(Path, extra_roots)]\\n","\\n","\\n","    def discover_datasets(extra_roots=()):\\n","        found = {}\\n","        unknown = set(DATASET_SOURCE_OVERRIDES) - set(DATASET_FOLDERS)\\n","        if unknown:\\n","            raise ValueError(f\'Unknown DATASET_SOURCE_OVERRIDES keys: {sorted(unknown)}\')\\n","        for key in SELECTED_DATASETS:\\n","            if key not in DATASET_SOURCE_OVERRIDES:\\n","                continue\\n","            path = Path(DATASET_SOURCE_OVERRIDES[key]).expanduser()\\n","            absent = [name for name in REQUIRED_FILES if not (path / name).is_file()]\\n","            if absent:\\n","                raise FileNotFoundError(f\'Incomplete explicit dataset source for {key}: {path}; missing {absent}\')\\n","            found[key] = path\\n","        aliases = {\'baby\': (\'baby\',), \'sports\': (\'sport\',), \'electronics\': (\'electronic\',)}\\n","        visited = set()\\n","        for root in dataset_search_roots(extra_roots):\\n","            root = root.expanduser()\\n","            if not root.is_dir() or str(root.resolve()) in visited:\\n","                continue\\n","            visited.add(str(root.resolve()))\\n","            for current, directories, files in os.walk(root, followlinks=False):\\n","                directories[:] = sorted(d for d in directories if d not in\\n","                                        {\'.git\', \'chunks\', \'__pycache__\', \'.stair_icaart_graph_cache\'})\\n","                if not set(REQUIRED_FILES).issubset(files):\\n","                    continue\\n","                path = Path(current)\\n","                for key in SELECTED_DATASETS:\\n","                    if any(alias in path.name.lower() for alias in aliases[key]):\\n","                        found.setdefault(key, path)\\n","        return found\\n","\\n","\\n","    def missing_dataset_message(missing):\\n","        return (\\n","            f\'Missing processed datasets: {sorted(missing)}.\\\\n\'\\n","            f\'On Kaggle, attach {DATASET_HANDLE!r} with Add Input BEFORE Save & Run All \'\\n","            \'(or attach an equivalent processed dataset matching the seed-1 fingerprints). \'\\n","            \'Save a new notebook version after attaching it. Internet access alone does not \'\\n","            \'attach a datasource, and kagglehub cannot attach a new one in a non-interactive run.\\\\n\'\\n","            \'Alternatively set DATASET_SOURCE_OVERRIDES to the existing per-dataset folders, \'\\n","            \'or DATASET_SEARCH_ROOTS to their parent directory. Each dataset folder must contain: \'\\n","            + \', \'.join(REQUIRED_FILES) + \'.\\\\n\'\\n","            \'Searched roots: \' + \', \'.join(str(p.expanduser()) for p in dataset_search_roots())\\n","            + f\'\\\\nDiscovery receipt: {RUN_ROOT / \\"provenance\\" / \\"data_discovery.json\\"}\'\\n","        )\\n","\\n","\\n","    def resolve_dataset_sources():\\n","        sources = discover_datasets()\\n","        missing = set(SELECTED_DATASETS) - sources.keys()\\n","        on_kaggle = Path(\'/kaggle/working\').is_dir()\\n","        discovery = {\'dataset_handle\': DATASET_HANDLE, \'selected_datasets\': list(SELECTED_DATASETS),\\n","                     \'searched_roots\': [str(p.expanduser()) for p in dataset_search_roots()],\\n","                     \'on_kaggle\': on_kaggle, \'auto_download_requested\': AUTO_DOWNLOAD_DATA,\\n","                     \'download_attempted\': False}\\n","\\n","        def record(status):\\n","            discovery.update(status=status, missing=sorted(missing),\\n","                             sources={key: str(path.resolve()) for key, path in sources.items()})\\n","            write_json(RUN_ROOT / \'provenance\' / \'data_discovery.json\', discovery)\\n","\\n","        if missing and on_kaggle:\\n","            # Never try to attach a datasource during a Kaggle batch job. The caller\\n","            # must attach inputs before execution; this also makes batch runs repeatable.\\n","            record(\'MISSING_ATTACHED_INPUT\')\\n","            raise FileNotFoundError(missing_dataset_message(missing))\\n","        if missing and AUTO_DOWNLOAD_DATA:\\n","            discovery[\'download_attempted\'] = True\\n","            try:\\n","                import kagglehub\\n","                downloaded = kagglehub.dataset_download(DATASET_HANDLE)\\n","                discovery[\'downloaded_root\'] = str(downloaded)\\n","                sources.update(discover_datasets([downloaded]))\\n","                missing = set(SELECTED_DATASETS) - sources.keys()\\n","            except Exception as exc:\\n","                discovery[\'download_error\'] = {\'type\': type(exc).__name__, \'message\': str(exc)}\\n","                record(\'DOWNLOAD_FAILED\')\\n","                raise FileNotFoundError(missing_dataset_message(missing)) from exc\\n","        if missing:\\n","            record(\'MISSING_PROCESSED_DATA\')\\n","            raise FileNotFoundError(missing_dataset_message(missing))\\n","        record(\'FOUND_PENDING_FINGERPRINT_CHECK\')\\n","        return sources\\n","\\n","\\n","    sources = resolve_dataset_sources()\\n","\\n","    DATA_RECEIPTS = {}\\n","    for key in SELECTED_DATASETS:\\n","        source = sources[key].resolve()\\n","        reference = SEED1_REFERENCE[key]\\n","        expected = dict(reference[\'contract\'][\'splits\'])\\n","        expected.update({record[\'file\']: record[\'sha256\'] for record in reference[\'features\']})\\n","        destination = DATA_ROOT / \'Processed\' / DATASET_FOLDERS[key]\\n","        if destination.is_symlink():\\n","            raise RuntimeError(f\'Processed destination must be a writable directory, not a symlink: {destination}\')\\n","        destination.mkdir(parents=True, exist_ok=True)\\n","        for name, expected_hash in expected.items():\\n","            original = source / name\\n","            if not original.is_file():\\n","                raise FileNotFoundError(original)\\n","            actual_hash = file_sha256(original)\\n","            if actual_hash != expected_hash:\\n","                raise RuntimeError(f\'Dataset differs from seed 1: {original}\')\\n","            target = destination / name\\n","            if target.is_symlink() and not target.exists():\\n","                raise FileExistsError(f\'Broken input link preserved; choose a fresh DATA_ROOT: {target}\')\\n","            if target.exists():\\n","                if file_sha256(target) != expected_hash:\\n","                    raise RuntimeError(f\'Existing linked/copied input has a different fingerprint: {target}\')\\n","            else:\\n","                try:\\n","                    target.symlink_to(original)\\n","                except OSError:\\n","                    shutil.copy2(original, target)\\n","        if (destination / \'chunks\').is_symlink():\\n","            raise RuntimeError(f\'FreeRec chunks must be writable, not an input symlink: {destination / \\"chunks\\"}\')\\n","        (destination / \'chunks\').mkdir(exist_ok=True)\\n","        DATA_RECEIPTS[key] = {\'source\': str(source), \'processed\': str(destination), \'fingerprints\': expected}\\n","        print(f\'{DATASET_TITLES[key]}: split and feature fingerprints match seed 1.\')\\n","    write_json(RUN_ROOT / \'provenance\' / \'data_check.json\', DATA_RECEIPTS)\\n","\\n","else:\\n","    print(\'report_only: skipped training setup; no model/network/GPU invocation.\')\\n"],"execution_count":null,"outputs":[]},{"cell_type":"markdown","id":"ad54d98c","metadata":{},"source":["## 5. YAML and resolved CLI preflight — only eta changes; no model constructed\\n"]},{"cell_type":"code","id":"e263ccdd","metadata":{},"source":["if RUN_MODE == \'train\':\\n","    import yaml\\n","\\n","    CONFIG_PATHS = {}\\n","    ARTIFACT_PATHS = {}\\n","    LOG_PATHS = {}\\n","    for key in SELECTED_DATASETS:\\n","        title = key.capitalize()\\n","        original = REPO_ROOT / \'configs\' / f\'Amazon2014{title}_STAIR_PEEC.yaml\'\\n","        if not original.is_file():\\n","            raise FileNotFoundError(original)\\n","        config = yaml.safe_load(original.read_text(encoding=\'utf-8\'))\\n","        config.update(SEED1_REFERENCE[key][\'v8_config\'])\\n","        config.update(SEED1_REFERENCE[key][\'icaart_config\'])\\n","        config[\'eta\'] = 0.0\\n","        config[\'evidence_delta\'] = 1.0\\n","        config.pop(\'evidence_seed\', None)  # Inactive; do not confuse the evidence partition with training RNG.\\n","        config.update(seed=SEED, root=str(DATA_ROOT), dataset=DATASET_FOLDERS[key],\\n","                      num_workers=SEED1_REFERENCE[key][\'contract\'][\'config\'][\'num_workers\'],\\n","                      eval_valid=True, eval_test=False, ranking=\'full\', which4best=\'NDCG@20\',\\n","                      resume=False, resume_from=\'\')\\n","        config[\'mfiles\'] = \',\'.join(config[\'mfiles\'])\\n","        config[\'num_neighbors\'] = \'-\'.join(map(str, config[\'num_neighbors\']))\\n","        artifact = RUN_ROOT / \'artifacts\' / STEM\\n","        config[\'artifact_dir\'] = str(artifact)\\n","        config[\'graph_cache_dir\'] = str(GRAPH_CACHE_ROOT / key)\\n","        target = RUN_ROOT / \'configs\' / f\'{STEM}.yaml\'\\n","        target.write_text(yaml.safe_dump(config, sort_keys=False), encoding=\'utf-8\')\\n","        shutil.copy2(original,RUN_ROOT/\\"configs\\"/f\\"{STEM}_original.yaml\\")\\n","        CONFIG_PATHS[key], ARTIFACT_PATHS[key] = target, artifact\\n","        LOG_PATHS[key] = RUN_ROOT / \'logs\' / f\'{STEM}.log\'\\n","\\n","\\n","    def build_command(key):\\n","        if key not in SELECTED_DATASETS:\\n","            raise ValueError(f\'{key} is not selected; edit SELECTED_DATASETS and rerun setup.\')\\n","        return [sys.executable, \'-u\', str(REPO_ROOT / \'main_stair_icaart.py\'),\\n","                \'--config\', str(CONFIG_PATHS[key]), \'--seed\', str(SEED),\\n","                \'--root\', str(DATA_ROOT), \'--dataset\', DATASET_FOLDERS[key],\\n","                \'--device\', \'0\', \'--ranking\', \'full\',\\n","                \'--artifact-dir\', str(ARTIFACT_PATHS[key]),\\n","                \'--graph-cache-dir\', str(GRAPH_CACHE_ROOT / key)]\\n","\\n","\\n","    config_probe = r\\"\\"\\"\\n","    import json, sys\\n","    from main_stair_icaart import build_config\\n","    cfg = build_config(sys.argv[1:])\\n","    keys = json.loads(__import__(\'os\').environ[\'ECLIPSE_CHECK_KEYS\'])\\n","    receipt = {key: getattr(cfg, key) for key in keys}\\n","    print(\'ECLIPSE_CONFIG=\' + json.dumps(receipt))\\n","    \\"\\"\\"\\n","    config_probe = __import__(\'textwrap\').dedent(config_probe)\\n","    for key in SELECTED_DATASETS:\\n","        expected = dict(SEED1_REFERENCE[key][\'v8_config\'])\\n","        expected.update(SEED1_REFERENCE[key][\'icaart_config\'])\\n","        expected[\'eta\'] = 0.0\\n","        expected[\'evidence_delta\'] = 1.0\\n","        expected.update(seed=SEED, dataset=DATASET_FOLDERS[key], ranking=\'full\', which4best=\'NDCG@20\',\\n","                        eval_valid=True, eval_test=False, resume=False, resume_from=\'\',\\n","                        num_workers=SEED1_REFERENCE[key][\'contract\'][\'config\'][\'num_workers\'])\\n","        probe_env = ENV.copy()\\n","        probe_env[\'ECLIPSE_CHECK_KEYS\'] = json.dumps(list(expected))\\n","        result = subprocess.run([sys.executable, \'-c\', config_probe, *build_command(key)[3:]],\\n","                                cwd=REPO_ROOT, env=probe_env, text=True, encoding=\'utf-8\', capture_output=True)\\n","        if result.returncode:\\n","            print(result.stdout, result.stderr)\\n","            result.check_returncode()\\n","        payload = next(line.split(\'=\', 1)[1] for line in result.stdout.splitlines()\\n","                       if line.startswith(\'ECLIPSE_CONFIG=\'))\\n","        resolved = json.loads(payload)\\n","        changes = {name: {\'expected\': value, \'actual\': resolved.get(name)}\\n","                   for name, value in expected.items() if resolved.get(name) != value}\\n","        if changes:\\n","            raise RuntimeError(f\'{key}: resolved CLI/YAML protocol mismatch: {changes}\')\\n","        write_json(RUN_ROOT / \'provenance\' / f\'{key}_resolved_config.json\', resolved)\\n","        print(f\'{key}: seed={resolved[\\"seed\\"]}, epochs={resolved[\\"epochs\\"]}, \'\\n","              f\'batch={resolved[\\"batch_size\\"]}, gamma={resolved[\\"gamma\\"]}, \'\\n","              f\'weight_decay={resolved[\\"weight_decay\\"]}, cl_chunk={resolved[\\"cl_chunk_size\\"]}\')\\n","    write_json(RUN_ROOT / \'run_manifest.json\', {\\n","        \'method\': \'ECLIPSE\', \'arm\': ARM, \'arm_id\': ARM, \'status\':\'READY\', \'training_seed\': SEED,\\n","        \'datasets\': list(SELECTED_DATASETS), \'session_id\': SESSION_ID,\\n","        \'reference_commit\': RUNTIME_COMMIT, \'actual_commit\': ACTUAL_COMMIT,\\n","        \'runtime\': RUNTIME_ENV, \'runtime_deviations\': deviations,\\n","        \'selection\': \'validation NDCG@20\', \'test_policy\': \'report only VALID-selected TEST; native terminal TEST may also be evaluated\',\\n","        \'configs\': {k: {\'path\': str(p), \'sha256\': file_sha256(p)} for k, p in CONFIG_PATHS.items()},\\n","        \'dataset_paths\': DATA_RECEIPTS,\\n","        \'note\': \'w/o CSE only: eta=0; PEEC delta1 and LC .01 retained; no resume.\',\\n","    })\\n","\\n","else:\\n","    print(\'report_only: skipped training setup; no model/network/GPU invocation.\')\\n"],"execution_count":null,"outputs":[]},{"cell_type":"markdown","id":"2aa0c8b3","metadata":{},"source":["## 6. Artifact validators — semantic/evidence invariants, not Full final-graph equality\\n","Checks eta0 in effective metadata, PEEC/LC retained, input/source contracts and native selected TEST. Historical source/runtime differences are recorded for human review. Diagnostics use an uncalibrated **semantic-only eta0** reference.\\n"]},{"cell_type":"code","id":"25f51837","metadata":{},"source":["import math\\n","import re\\n","import signal\\n","import time\\n","import pandas as pd\\n","import numpy as np\\n","from IPython.display import display\\n","\\n","RESULTS = {}\\n","\\n","\\n","def read_jsonl(path):\\n","    path = Path(path)\\n","    if not path.is_file():\\n","        return []\\n","    result = []\\n","    with path.open(encoding=\'utf-8\') as handle:\\n","        for number, line in enumerate(handle, 1):\\n","            if line.strip():\\n","                try:\\n","                    result.append(json.loads(line))\\n","                except json.JSONDecodeError as exc:\\n","                    raise ValueError(f\'Malformed JSONL: {path}, line {number}\') from exc\\n","    return result\\n","\\n","\\n","def compare_manifest(key, manifest, expected_seed):\\n","    ref=SEED1_REFERENCE[key];problems={}\\n","    if manifest.get(\'seed\')!=expected_seed:problems[\'seed\']=manifest.get(\'seed\')\\n","    if manifest.get(\'selection\')!=\'validation NDCG@20\':problems[\'selection\']=manifest.get(\'selection\')\\n","    for name,expected in dict(ref[\'v8_config\'],eta=0.0).items():\\n","        if manifest.get(\'v8_config\',{}).get(name)!=expected:problems[\'v8_config.\'+name]=manifest.get(\'v8_config\',{}).get(name)\\n","    if manifest.get(\'icaart_config\')!=ref[\'icaart_config\']:problems[\'evidence_config\']=manifest.get(\'icaart_config\')\\n","    contract=manifest.get(\'checkpoint_contract\',{})\\n","    for name in (\'splits\',\'candidate_hashes\',\'data_fingerprint\'):\\n","        if contract.get(name)!=ref[\'contract\'][name]:problems[name]=contract.get(name)\\n","    for name,expected in ref[\'contract\'][\'config\'].items():\\n","        if name==\'seed\':expected=expected_seed\\n","        if name==\'eta\':expected=0.0\\n","        if contract.get(\'config\',{}).get(name)!=expected:problems[\'config.\'+name]=contract.get(\'config\',{}).get(name)\\n","    if manifest.get(\'features\')!=ref[\'features\']:problems[\'features\']=manifest.get(\'features\')\\n","    if manifest.get(\'precision\')!={\'tf32\':False,\'amp\':False}:problems[\'precision\']=manifest.get(\'precision\')\\n","    e=manifest.get(\'icaart_manifest\',{});g=manifest.get(\'graph\',{});d=g.get(\'icaart\',{})\\n","    for name,expected in ref[\'evidence_invariants\'].items():\\n","        if e.get(name)!=expected:problems[\'evidence.\'+name]=e.get(name)\\n","    if e.get(\'training_seed\')!=expected_seed:problems[\'evidence.training_seed\']=e.get(\'training_seed\')\\n","    for name,actual in ((\'graph.eta\',g.get(\'eta\')),(\'graph.effective_eta\',g.get(\'effective_eta\')),(\'icaart.effective_eta\',d.get(\'effective_eta\'))):\\n","        if actual!=0.0:problems[name]=actual\\n","    if e.get(\'null_scope\')!=\'full_activity\' or e.get(\'normalization\')!=\'actual_degree\':problems[\'null_normalization\']=e\\n","    fp=contract.get(\'graph_fingerprint\')\\n","    if not isinstance(fp,str) or len(fp)!=64:problems[\'final_graph_fingerprint\']=fp\\n","    for name,actual in ((\'graph\',g.get(\'graph_fingerprint\')),(\'icaart_manifest\',e.get(\'graph_fingerprint\')),(\'diagnostics\',d.get(\'s_ica_fingerprint\'))):\\n","        if actual!=fp:problems[\'final_graph_consistency.\'+name]=actual\\n","    if problems:raise RuntimeError(f\'{key}: w/o CSE comparability failed: {problems}\')\\n","    source_diff={k:{\'reference\':v,\'actual\':contract.get(\'sources\',{}).get(k)} for k,v in ref[\'contract\'][\'sources\'].items() if contract.get(\'sources\',{}).get(k)!=v}\\n","    runtime_diff={k:{\'reference\':ref[k],\'actual\':manifest.get(k)} for k in (\'torch\',\'freerec\',\'gpu\') if manifest.get(k)!=ref[k]}\\n","    if str(manifest.get(\'python\',\'\')).split(\'.\')[:2]!=ref[\'python\'].split(\'.\')[:2]:runtime_diff[\'python\']=manifest.get(\'python\')\\n","    if RUN_MODE==\'train\' and source_diff:raise RuntimeError(f\'Source mismatch: {source_diff}\')\\n","    if RUN_MODE==\'train\' and STRICT_RUNTIME and runtime_diff:raise RuntimeError(f\'Runtime mismatch: {runtime_diff}\')\\n","    return {\'status\':\'MATCH_WITH_REVIEW_REQUIRED\' if source_diff or runtime_diff else \'MATCH\',\\n","        \'arm\':ARM,\'training_seed\':expected_seed,\'model_intervention\':{\'eta\':[0.1,0.0]},\\n","        \'same_candidates_semantic_evidence\':True,\'same_final_graph_as_full\':\'NOT_REQUIRED\',\\n","        \'dose_reference\':\'uncalibrated semantic-only eta0\',\'source_deviations\':source_diff,\'runtime_deviations\':runtime_diff,\\n","        \'operator_check_scope\':\'manifest and pinned source contract; not an independent sparse-tensor reconstruction\',\\n","        \'cf_preprocessing_may_remain\':True}\\n","\\n","\\n","def validate_history(telemetry, evaluations, receipt):\\n","    if [row.get(\'epoch\') for row in telemetry]!=list(range(1,501)):raise ValueError(\'Require exactly ordered epochs1..500; no duplicate/missing epochs or implicit resume\')\\n","    for row in telemetry:\\n","        for name in (\'loss\',\'bpr_loss\',\'nlgcl_loss\',\'weighted_nlgcl_loss\',\'seconds\',\'peak_allocated_gib\',\'peak_reserved_gib\'):\\n","            value=row.get(name)\\n","            if isinstance(value,bool) or not isinstance(value,(int,float)) or not math.isfinite(value) or value<0:raise ValueError(f\'Invalid {name} at epoch {row[\\"epoch\\"]}\')\\n","        if row.get(\'lambda_nlgcl\')!=.01:raise ValueError(\'LC must stay enabled\')\\n","        if not math.isclose(row[\'weighted_nlgcl_loss\'],.01*row[\'nlgcl_loss\'],rel_tol=1e-6,abs_tol=1e-7):raise ValueError(\'Weighted LC mismatch\')\\n","        if not math.isclose(row[\'loss\'],row[\'bpr_loss\']+row[\'weighted_nlgcl_loss\'],rel_tol=1e-6,abs_tol=1e-7):raise ValueError(\'Objective mismatch\')\\n","    valid=[r for r in evaluations if r.get(\'mode\')==\'valid\' and not r.get(\'selected_checkpoint\')]\\n","    if not valid:raise ValueError(\'Missing VALID history\')\\n","    seen=set()\\n","    for row in valid:\\n","        epoch=row.get(\'epoch\');v=row.get(\'metrics\',{}).get(\'NDCG@20\')\\n","        if isinstance(epoch,bool) or not isinstance(epoch,int) or not 0<=epoch<=500 or epoch in seen:raise ValueError(\'Invalid/duplicate VALID epoch\')\\n","        seen.add(epoch)\\n","        if isinstance(v,bool) or not isinstance(v,(int,float)) or not math.isfinite(v):raise ValueError(\'Invalid VALID NDCG@20\')\\n","    maximum=max(r[\'metrics\'][\'NDCG@20\'] for r in valid)\\n","    tied=[r[\'epoch\'] for r in valid if r[\'metrics\'][\'NDCG@20\']==maximum]\\n","    if receipt[\'epoch\'] not in tied:raise ValueError(\'Selected epoch is not a VALID maximum\')\\n","    selected=[r for r in evaluations if r.get(\'mode\')==\'test\' and r.get(\'selected_checkpoint\') is True]\\n","    if len(selected)!=1 or selected[0]!=receipt:raise ValueError(\'Selected receipt/history mismatch\')\\n","    return {\'valid_max_epochs\':tied,\'tie_rule\':\'native receipt retained; inspect native log if tied\' if len(tied)>1 else \'unique maximum\',\\n","        \'test_events\':[{\'epoch\':r.get(\'epoch\'),\'selected\':r.get(\'selected_checkpoint\')} for r in evaluations if r.get(\'mode\')==\'test\']}\\n","\\n","\\n","def validate_receipt(key, receipt, manifest, expected_seed):\\n","    if (receipt.get(\'mode\') != \'test\' or receipt.get(\'selected_checkpoint\') is not True\\n","            or receipt.get(\'selection\') != \'validation NDCG@20\'\\n","            or receipt.get(\'seed\') != expected_seed\\n","            or receipt.get(\'evidence_mode\') != \'pooled_activity_excess\'\\n","            or receipt.get(\'evidence_delta\') != 1.0):\\n","        raise ValueError(f\'{key}: not a selected-checkpoint ECLIPSE w/o CSE TEST receipt\')\\n","    if receipt.get(\'graph_fingerprint\') != manifest.get(\'checkpoint_contract\', {}).get(\'graph_fingerprint\'):\\n","        raise ValueError(f\'{key}: metric receipt belongs to another operator\')\\n","    epoch = receipt.get(\'epoch\')\\n","    if isinstance(epoch, bool) or not isinstance(epoch, int) or not 1 <= epoch <= 500:\\n","        raise ValueError(f\'{key}: invalid selected epoch\')\\n","    normalized = receipt.get(\'metrics\', {})\\n","    for metric in PRIMARY_METRICS:\\n","        value = normalized.get(metric)\\n","        if isinstance(value, bool) or not isinstance(value, (float, int)) or not math.isfinite(value) or not 0 <= value <= 1:\\n","            raise ValueError(f\'{key}: missing/invalid measured TEST metric {metric}\')\\n","    return normalized\\n","\\n","\\n","def export_run_tables(key):\\n","    artifact = ARTIFACT_PATHS[key]\\n","    manifest = json.loads((artifact / \'manifest.json\').read_text(encoding=\'utf-8\'))\\n","    status = json.loads((artifact / \'run_status.json\').read_text(encoding=\'utf-8\'))\\n","    if status.get(\'status\') != \'COMPLETED\' or status.get(\'seed\') != SEED:\\n","        raise RuntimeError(f\'{key}: run is not completed for training seed {SEED}\')\\n","    comparability = compare_manifest(key, manifest, SEED)\\n","    receipt = json.loads((artifact / \'selected_metrics.json\').read_text(encoding=\'utf-8\'))\\n","    metrics = validate_receipt(key, receipt, manifest, SEED)\\n","    telemetry = pd.DataFrame(read_jsonl(artifact / \'training_telemetry.jsonl\'))\\n","    evaluations = read_jsonl(artifact / \'evaluation_metrics.jsonl\')\\n","    history = validate_history(telemetry.to_dict(\'records\'), evaluations, receipt)\\n","    checkpoint = artifact / \'best_model.pt\'\\n","    if not checkpoint.is_file(): raise FileNotFoundError(checkpoint)\\n","    telemetry.to_csv(RUN_ROOT / \'tables\' / f\'{STEM}_training_telemetry.csv\', index=False, na_rep=\'NOT_MEASURED\')\\n","    evaluation_rows = [{k: v for k, v in record.items() if k != \'metrics\'} |\\n","                       {str(k).upper(): v for k, v in record.get(\'metrics\', {}).items()}\\n","                       for record in evaluations]\\n","    pd.DataFrame(evaluation_rows).to_csv(RUN_ROOT / \'tables\' / f\'{STEM}_evaluations.csv\',\\n","                                       index=False, na_rep=\'NOT_MEASURED\')\\n","    log_text = LOG_PATHS[key].read_text(encoding=\'utf-8\', errors=\'replace\') if LOG_PATHS[key] and LOG_PATHS[key].is_file() else \'\'\\n","    fit_matches = re.findall(r\'Coach\\\\.fit takes\\\\s+([0-9.eE+-]+)\\\\s+seconds\', log_text)\\n","    summary = {\'method\': \'ECLIPSE\', \'arm\': ARM, \'dataset\': key, \'seed\': SEED,\\n","               \'selected_epoch\': receipt[\'epoch\'], \'split\': \'TEST\', **metrics,\\n","               \'fit_seconds\': float(fit_matches[-1]) if fit_matches else None,\\n","               \'train_epoch_seconds_sum\': float(telemetry[\'seconds\'].sum()),\\n","               \'preprocessing_seconds\': manifest.get(\'preprocessing_seconds\'),\\n","               \'peak_training_allocated_mib\': float(telemetry[\'peak_allocated_gib\'].max() * 1024),\\n","               \'peak_training_reserved_mib\': float(telemetry[\'peak_reserved_gib\'].max() * 1024),\\n","               \'memory_scope\': \'child-process per-training-epoch CUDA allocator peaks; excludes evaluation peaks\',\\n","               \'commit\': manifest.get(\'commit\'), \'graph_fingerprint\': receipt[\'graph_fingerprint\'],\\n","               \'artifact_dir\': str(artifact), \'checkpoint_path\': str(artifact / \'best_model.pt\'),\\n","               \'runtime_deviations\': comparability[\'runtime_deviations\'], \'comparability\': comparability}\\n","    write_json(RUN_ROOT / \'tables\' / f\'{STEM}_selected_test.json\', summary)\\n","    write_json(RUN_ROOT / \'tables\' / f\'{STEM}_mechanism.json\', manifest[\'graph\'][\'icaart\'])\\n","    execution_path=RUN_ROOT/\'provenance\'/f\'{STEM}_notebook_execution.json\'\\n","    execution=json.loads(execution_path.read_text(encoding=\'utf-8\')) if execution_path.is_file() else {}\\n","    summary.update(status=\'COMPLETED_PENDING_HUMAN_REVIEW\', attempt_id=SESSION_ID,\\n","       checkpoint_sha256=file_sha256(checkpoint), history=history,\\n","       subprocess_wall_seconds=execution.get(\'subprocess_wall_seconds\'),\\n","       preprocessing_cache_hit=manifest.get(\'graph\',{}).get(\'cache_hit\'), reviewed_by=None)\\n","    pd.DataFrame([{k:v for k,v in summary.items() if not isinstance(v,(dict,list))}]).to_csv(\\n","        RUN_ROOT/\'tables\'/f\'{STEM}_selected_test.csv\',index=False,na_rep=\'NOT_MEASURED\')\\n","    write_json(RUN_ROOT/\'tables\'/f\'{STEM}_selected_test.json\',summary)\\n","    write_json(RUN_ROOT/\'provenance\'/f\'{STEM}_comparability.json\',comparability)\\n","    write_json(RUN_ROOT/\'provenance\'/f\'{STEM}_input_receipts.json\',{\\n","       name:file_sha256(artifact/name) for name in (\'manifest.json\',\'run_status.json\',\'selected_metrics.json\',\'training_telemetry.jsonl\',\'evaluation_metrics.jsonl\',\'best_model.pt\')})\\n","    RESULTS[key] = summary\\n","    return summary\\n","\\n","\\n","def train_dataset(key):\\n","    if RUN_MODE == \'report_only\': return export_run_tables(key)\\n","    if key not in SELECTED_DATASETS:\\n","        print(f\'Skipped {key}: not in SELECTED_DATASETS.\')\\n","        return None\\n","    artifact = ARTIFACT_PATHS[key]\\n","    status_path = artifact / \'run_status.json\'\\n","    if status_path.is_file():\\n","        status = json.loads(status_path.read_text(encoding=\'utf-8\'))\\n","        if status.get(\'status\') == \'COMPLETED\':\\n","            print(f\'{key}: reading the completed attempt; no retraining.\')\\n","            return export_run_tables(key)\\n","    # build_config() creates artifact_dir during CLI preflight, before training.\\n","    # An empty real directory with no stdout log is not a previous run.\\n","    log_path = LOG_PATHS[key]\\n","    invalid_directory = artifact.is_symlink() or (artifact.exists() and not artifact.is_dir())\\n","    has_payload = artifact.is_dir() and any(artifact.iterdir())\\n","    has_log = log_path.exists() or log_path.is_symlink()\\n","    if invalid_directory or has_payload or has_log:\\n","        raise FileExistsError(f\'Existing attempt payload/log preserved at {artifact}; \'\\n","                              \'use report_only for completed runs, or create a new SESSION_ID for a fresh attempt. \'\\n","                              \'No automatic resume/delete/retry.\')\\n","    artifact.mkdir(parents=True, exist_ok=True)\\n","    command = build_command(key)\\n","    write_json(RUN_ROOT / \'provenance\' / f\'{STEM}_command.json\', {\\n","        \'argv\': command, \'training_seed\': SEED, \'arm_id\': ARM, \'dataset\': key, \'purpose\':\'component_ablation\', \'pythonhashseed\': ENV[\'PYTHONHASHSEED\'],\\n","        \'cuda_visible_devices\': ENV[\'CUDA_VISIBLE_DEVICES\'], \'cwd\': str(REPO_ROOT)})\\n","    print(\'Running:\', \' \'.join(command))\\n","    start = time.perf_counter()\\n","    manifest_checked = False\\n","    process = None\\n","    try:\\n","        with LOG_PATHS[key].open(\'x\', encoding=\'utf-8\') as log:\\n","            process = subprocess.Popen(command, cwd=REPO_ROOT, env=ENV, stdout=subprocess.PIPE,\\n","                                       stderr=subprocess.STDOUT, text=True, encoding=\'utf-8\',\\n","                                       errors=\'replace\', bufsize=1)\\n","            for line in process.stdout:\\n","                log.write(line)\\n","                log.flush()\\n","                print(line, end=\'\', flush=True)\\n","                manifest_path = artifact / \'manifest.json\'\\n","                if not manifest_checked and manifest_path.is_file():\\n","                    comparability = compare_manifest(key, json.loads(manifest_path.read_text(encoding=\'utf-8\')), SEED)\\n","                    write_json(RUN_ROOT / \'provenance\' / f\'{key}_comparability.json\', comparability)\\n","                    manifest_checked = True\\n","            process.wait()\\n","            process.stdout.close()\\n","        if process.returncode != 0:\\n","            raise RuntimeError(f\'{key}: training failed with exit code {process.returncode}; see {LOG_PATHS[key]}\')\\n","        child_wall_seconds=time.perf_counter()-start\\n","        write_json(RUN_ROOT/\'provenance\'/f\'{STEM}_notebook_execution.json\',{\'status\':\'CHILD_COMPLETED_PENDING_EXPORT\',\'subprocess_wall_seconds\':child_wall_seconds})\\n","        result = export_run_tables(key)\\n","        write_json(RUN_ROOT / \'provenance\' / f\'{STEM}_notebook_execution.json\', {\\n","            \'status\': \'COMPLETED\', \'subprocess_wall_seconds\': child_wall_seconds,\\n","            \'exit_code\': process.returncode, \'training_seed\': SEED})\\n","        print(f\'\\\\nCompleted {DATASET_TITLES[key]} | selected TEST NDCG@20={result[\\"NDCG@20\\"]:.10f}\')\\n","        return result\\n","    except BaseException as exc:\\n","        if process is not None and process.poll() is None:\\n","            if isinstance(exc, KeyboardInterrupt):\\n","                process.send_signal(signal.SIGINT)\\n","            else:\\n","                process.terminate()\\n","            try:\\n","                process.wait(timeout=20)\\n","            except subprocess.TimeoutExpired:\\n","                process.kill()\\n","                process.wait(timeout=10)\\n","        write_json(RUN_ROOT / \'provenance\' / f\'{STEM}_notebook_execution.json\', {\\n","            \'status\': \'INTERRUPTED\' if isinstance(exc, KeyboardInterrupt) else \'FAILED\',\\n","            \'error_type\': type(exc).__name__, \'error\': str(exc), \'training_seed\': SEED,\\n","            \'subprocess_wall_seconds\': time.perf_counter() - start})\\n","        raise\\n"],"execution_count":null,"outputs":[]},{"cell_type":"markdown","id":"b430b2e9","metadata":{},"source":["## 7. Execute one fresh run or report-only export\\n","This is the only training invocation. Reexecution of a completed attempt exports instead of training; incomplete attempts are preserved. Interrupt/failure cleans up only the child process launched by this cell. No automatic resume/retry.\\n"]},{"cell_type":"code","id":"2a856b3c","metadata":{},"source":["result=train_dataset(\'electronics\')\\n","display(pd.DataFrame([{k:result[k] for k in (\'arm\',\'dataset\',\'seed\',\'selected_epoch\',*PRIMARY_METRICS)}]))\\n"],"execution_count":null,"outputs":[]},{"cell_type":"markdown","id":"24cfe764","metadata":{},"source":["## 8. Paper figures — PDF vector, PNG300dpi, source CSV/JSON and English captions\\n","No synthetic curves, no TEST-based epoch selection. Resource data refer to the child process, not parent-kernel GPU measurements. Calibration stays active in w/o CSE; reference is semantic-only eta0.\\n"]},{"cell_type":"code","id":"9e5ec005","metadata":{},"source":["import matplotlib.pyplot as plt\\n","from matplotlib.patches import FancyBboxPatch\\n","\\n","plt.rcParams.update({\'font.family\': \'DejaVu Sans\', \'font.size\': 9, \'axes.titlesize\': 10,\\n","                     \'axes.labelsize\': 9, \'legend.fontsize\': 8, \'xtick.labelsize\': 8,\\n","                     \'ytick.labelsize\': 8, \'lines.linewidth\': 1.5,\\n","                     \'pdf.fonttype\': 42, \'ps.fonttype\': 42, \'savefig.dpi\': 300,\\n","                     \'axes.spines.top\': False, \'axes.spines.right\': False})\\n","COLORS = {\'blue\': \'#0072B2\', \'orange\': \'#D55E00\', \'green\': \'#009E73\',\\n","          \'purple\': \'#CC79A7\', \'gray\': \'#666666\'}\\n","_existing_figure_manifest = RUN_ROOT / \'figures\' / \'figure_manifest.json\'\\n","FIGURE_RECORDS = json.loads(_existing_figure_manifest.read_text(encoding=\'utf-8\')) if _existing_figure_manifest.is_file() else []\\n","\\n","\\n","def save_figure(fig, stem, caption, sources):\\n","    for extension in (\'pdf\', \'png\'):\\n","        fig.savefig(RUN_ROOT / \'figures\' / f\'{stem}.{extension}\', bbox_inches=\'tight\', facecolor=\'white\')\\n","    plt.show()\\n","    plt.close(fig)\\n","    FIGURE_RECORDS[:] = [record for record in FIGURE_RECORDS if record[\'stem\'] != stem]\\n","    FIGURE_RECORDS.append({\'stem\': stem, \'caption\': caption, \'sources\': list(map(str, sources))})\\n","    write_json(RUN_ROOT / \'figures\' / \'figure_manifest.json\', FIGURE_RECORDS)\\n","    (RUN_ROOT/\'figures\'/\'FIGURE_CAPTIONS.md\').write_text(\'# Figure captions\\\\n\\\\n\'+\'\\\\n\\\\n\'.join(\'## \'+r[\'stem\']+\'\\\\n\\\\n\'+r[\'caption\'] for r in FIGURE_RECORDS),encoding=\'utf-8\')\\n","    print(caption)\\n","\\n","\\n","def completed_artifact(key):\\n","    if key not in SELECTED_DATASETS:\\n","        print(f\'{key}: not selected; no figure generated.\')\\n","        return None\\n","    path = ARTIFACT_PATHS[key]\\n","    status = path / \'run_status.json\'\\n","    if not status.is_file() or json.loads(status.read_text(encoding=\'utf-8\')).get(\'status\') != \'COMPLETED\':\\n","        print(f\'{key}: no completed run available; no figure generated.\')\\n","        return None\\n","    return path\\n","\\n","\\n","def plot_learning_curves(key):\\n","    artifact = completed_artifact(key)\\n","    if artifact is None:\\n","        return\\n","    telemetry = pd.DataFrame(read_jsonl(artifact / \'training_telemetry.jsonl\'))\\n","    records = [record for record in read_jsonl(artifact / \'evaluation_metrics.jsonl\')\\n","               if record.get(\'mode\') == \'valid\' and not record.get(\'selected_checkpoint\')]\\n","    selected = json.loads((artifact / \'selected_metrics.json\').read_text(encoding=\'utf-8\'))[\'epoch\']\\n","    valid = pd.DataFrame([{\'epoch\': record[\'epoch\'],\\n","                           **{str(k).upper(): v for k, v in record[\'metrics\'].items()}}\\n","                          for record in records]).sort_values(\'epoch\')\\n","    if telemetry.empty or valid.empty:\\n","        print(f\'{key}: missing measured curves; no substitute data generated.\')\\n","        return\\n","    fig, axes = plt.subplots(1, 3, figsize=(7.2, 3.0), constrained_layout=True)\\n","    for column, label, color in [(\'loss\', \'Total\', COLORS[\'blue\']),\\n","                                  (\'bpr_loss\', \'BPR\', COLORS[\'orange\']),\\n","                                  (\'weighted_nlgcl_loss\', \'Weighted LC\', COLORS[\'green\'])]:\\n","        if column in telemetry:\\n","            axes[0].plot(telemetry[\'epoch\'], telemetry[column], label=label, color=color)\\n","    axes[0].set(xlabel=\'Training epoch\', ylabel=\'Mean training loss\', title=\'Training objective\')\\n","    axes[0].legend(frameon=False)\\n","    for ax, metric in zip(axes[1:], (\'NDCG@20\', \'RECALL@20\')):\\n","        if metric in valid:\\n","            ax.plot(valid[\'epoch\'], valid[metric], color=COLORS[\'blue\'], label=f\'VALID {metric}\')\\n","            point = valid.loc[valid[\'epoch\'] == selected, metric]\\n","            if len(point):\\n","                ax.scatter([selected], [point.iloc[-1]], color=COLORS[\'orange\'], s=22, zorder=3)\\n","        ax.axvline(selected, color=COLORS[\'gray\'], linestyle=\'--\', linewidth=1, label=f\'Selected epoch {selected}\')\\n","        ax.set(xlabel=\'Training epoch\', ylabel=f\'Validation {metric}\', title=f\'Validation {metric}\')\\n","        ax.legend(frameon=False)\\n","    for ax in axes:\\n","        ax.grid(alpha=.2)\\n","    fig.suptitle(f\'ECLIPSE w/o CSE | {DATASET_TITLES[key]} | seed {SEED}\', fontsize=11)\\n","    save_figure(fig, f\'{STEM}_learning_curves\',\\n","                \'Training objective and validation ranking curves. The checkpoint is selected by validation NDCG@20; \'\\n","                \'TEST is evaluated at the selected checkpoint and is not a learning-curve selection signal.\',\\n","                [artifact / \'training_telemetry.jsonl\', artifact / \'evaluation_metrics.jsonl\', artifact / \'selected_metrics.json\'])\\n","\\n","\\n","def plot_training_memory(key):\\n","    artifact = completed_artifact(key)\\n","    if artifact is None:\\n","        return\\n","    telemetry = pd.DataFrame(read_jsonl(artifact / \'training_telemetry.jsonl\'))\\n","    if telemetry.empty:\\n","        return\\n","    fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.8), constrained_layout=True)\\n","    for column, label, color in [(\'peak_allocated_gib\', \'Peak allocated\', COLORS[\'blue\']),\\n","                                  (\'peak_reserved_gib\', \'Peak reserved\', COLORS[\'orange\'])]:\\n","        if column in telemetry:\\n","            axes[0].plot(telemetry[\'epoch\'], telemetry[column] * 1024, label=label, color=color)\\n","    axes[0].set(xlabel=\'Training epoch\', ylabel=\'CUDA allocator memory (MiB)\', title=\'Training-epoch allocator peaks\')\\n","    axes[0].legend(frameon=False)\\n","    axes[1].plot(telemetry[\'epoch\'], telemetry[\'seconds\'], color=COLORS[\'green\'])\\n","    axes[1].set(xlabel=\'Training epoch\', ylabel=\'Training time (s/epoch)\', title=\'Training time per epoch\')\\n","    for ax in axes:\\n","        ax.grid(alpha=.2)\\n","    fig.suptitle(f\'ECLIPSE w/o CSE | {DATASET_TITLES[key]} | seed {SEED}\', fontsize=11)\\n","    save_figure(fig, f\'{STEM}_training_memory\',\\n","                \'CUDA allocator peaks are measured in the training subprocess for each training epoch. \'\\n","                \'They exclude evaluation peaks, other GPU processes and driver memory. No arbitrary hardware ceiling is drawn.\',\\n","                [artifact / \'training_telemetry.jsonl\'])\\n","\\n","\\n","def plot_calibration_diagnostics(key):\\n","    artifact = completed_artifact(key)\\n","    if artifact is None:\\n","        return\\n","    manifest = json.loads((artifact / \'manifest.json\').read_text(encoding=\'utf-8\'))\\n","    diagnostics = manifest[\'graph\'][\'icaart\']\\n","    coverage = [diagnostics.get(\'frac_edges_with_positive_evidence\'), diagnostics.get(\'weighted_evidence_coverage\')]\\n","    if any(value is None for value in coverage):\\n","        print(f\'{key}: coverage was not measured.\')\\n","        return\\n","    fig, axes = plt.subplots(1, 3, figsize=(7.2, 3.0), constrained_layout=True)\\n","    bars = axes[0].bar([\'Candidate\\\\nedges\', \'Raw semantic\\\\nweight\'], np.asarray(coverage) * 100,\\n","                       color=[COLORS[\'blue\'], COLORS[\'green\']], width=.6)\\n","    axes[0].bar_label(bars, fmt=\'%.2f%%\', padding=3, fontsize=8)\\n","    axes[0].set(ylabel=\'Coverage with positive evidence (%)\', title=\'Calibration coverage\')\\n","    axes[0].set_ylim(0, max(coverage) * 125 + 1)\\n","    polynomial = diagnostics.get(\'polynomial_dose_by_coordinate\', {})\\n","    if polynomial:\\n","        coordinates = sorted(map(int, polynomial))\\n","        values = [100 * polynomial[str(coordinate)] for coordinate in coordinates]\\n","        axes[1].plot(coordinates, values, \'o-\', color=COLORS[\'blue\'], label=\'Polynomial probe dose\')\\n","    operator_dose = diagnostics.get(\'relative_operator_dose\')\\n","    if operator_dose is not None:\\n","        axes[1].axhline(100 * operator_dose, color=COLORS[\'orange\'], linestyle=\'--\', label=\'Operator Frobenius dose\')\\n","    axes[1].set(xlabel=\'Embedding coordinate\', ylabel=\'Relative dose (%)\', title=\'Operator / polynomial\\\\nprobe dose\')\\n","    axes[1].legend(frameon=False)\\n","    levels, values = diagnostics.get(\'evidence_quantile_levels\'), diagnostics.get(\'evidence_score_quantiles\')\\n","    if levels is not None and values is not None and len(levels) == len(values):\\n","        axes[2].plot(np.asarray(levels) * 100, values, \'o-\', color=COLORS[\'green\'])\\n","        pd.DataFrame({\'quantile\': levels, \'evidence_score\': values}).to_csv(\\n","            RUN_ROOT / \'tables\' / f\'{STEM}_evidence_quantiles.csv\', index=False)\\n","    else:\\n","        axes[2].text(.5, .5, \'Score quantiles not measured\', ha=\'center\', va=\'center\', transform=axes[2].transAxes)\\n","    axes[2].set(xlabel=\'Quantile (%)\', ylabel=\'Excess-evidence score\', ylim=(0, 1), title=\'Evidence-score quantile profile\')\\n","    for ax in axes:\\n","        ax.grid(axis=\'y\', alpha=.2)\\n","    write_json(RUN_ROOT/\'tables\'/f\'{STEM}_calibration_figure_data.json\',{\\n","       \'coverage_fraction\':coverage,\'operator_relative_dose\':operator_dose,\'polynomial_dose_by_coordinate\':polynomial,\\n","       \'quantile_levels\':levels,\'score_quantiles\':values,\'reference\':\'uncalibrated semantic-only eta0\',\\n","       \'probe_scope\':diagnostics.get(\'polynomial_probe_scope\')})\\n","    fig.suptitle(f\'ECLIPSE w/o CSE | {DATASET_TITLES[key]} | seed {SEED}\', fontsize=11)\\n","    save_figure(fig, f\'{STEM}_calibration_diagnostics\',\\n","                \'w/o CSE: PEEC remains active; reference is the uncalibrated semantic-only eta0 operator. \'\\n","                \'Polynomial dose uses fixed Rademacher probes, not actual Adam directions. \'\\n","                \'The score panel displays measured quantiles, not a reconstructed histogram.\', [artifact / \'manifest.json\'])\\n"],"execution_count":null,"outputs":[]},{"cell_type":"markdown","id":"fb3a873a","metadata":{},"source":["## 9. TRAIN objective and VALID ranking curves\\n"]},{"cell_type":"code","id":"77ffd936","metadata":{},"source":["plot_learning_curves(\'electronics\')\\n"],"execution_count":null,"outputs":[]},{"cell_type":"markdown","id":"deefd0c4","metadata":{},"source":["## 10. Training allocator memory and epoch time\\n"]},{"cell_type":"code","id":"1ef91022","metadata":{},"source":["plot_training_memory(\'electronics\')\\n"],"execution_count":null,"outputs":[]},{"cell_type":"markdown","id":"df582e50","metadata":{},"source":["## 11. Calibration diagnostics — semantic-only reference\\n"]},{"cell_type":"code","id":"374cfd62","metadata":{},"source":["plot_calibration_diagnostics(\'electronics\')\\n"],"execution_count":null,"outputs":[]},{"cell_type":"markdown","id":"f65219e5","metadata":{},"source":["## 12. Selected TEST and resource summary — one measured run only\\n","No historical seeds/Full metrics are injected. Coach.fit comes from native stdout; missing log timing is NOT_MEASURED, not zero. Subprocess wall and summed training epoch time remain separate.\\n"]},{"cell_type":"code","id":"94b23b6d","metadata":{},"source":["summary_rows = []\\n","for key in SELECTED_DATASETS:\\n","    if completed_artifact(key) is None:\\n","        summary_rows.append({\'method\': \'ECLIPSE\', \'dataset\': key, \'seed\': SEED, \'status\': \'NOT_MEASURED\'})\\n","        continue\\n","    summary = export_run_tables(key)\\n","    names = [\'method\', \'arm\', \'dataset\', \'seed\', \'selected_epoch\', *PRIMARY_METRICS,\\n","             \'preprocessing_seconds\', \'fit_seconds\', \'train_epoch_seconds_sum\',\\n","             \'peak_training_allocated_mib\', \'peak_training_reserved_mib\',\\n","             \'checkpoint_path\', \'commit\', \'graph_fingerprint\']\\n","    summary_rows.append({name: summary.get(name) for name in names} | {\'status\': \'COMPLETED\'})\\n","SUMMARY = pd.DataFrame(summary_rows)\\n","for column in [\'selected_epoch\', *PRIMARY_METRICS, \'preprocessing_seconds\', \'fit_seconds\',\\n","               \'peak_training_allocated_mib\', \'peak_training_reserved_mib\']:\\n","    if column not in SUMMARY:\\n","        SUMMARY[column] = np.nan\\n","SUMMARY.to_csv(RUN_ROOT / \'tables\' / f\'{STEM}_selected_test_overview.csv\', index=False, na_rep=\'NOT_MEASURED\')\\n","display(SUMMARY[[\'dataset\', \'seed\', \'status\', \'selected_epoch\', *PRIMARY_METRICS]].fillna(\'NOT_MEASURED\'))\\n","measured = SUMMARY.loc[SUMMARY[\'status\'] == \'COMPLETED\'].copy()\\n","if len(measured):\\n","    labels = [DATASET_TITLES[key].replace(\'Amazon \', \'\') for key in measured[\'dataset\']]\\n","    x = np.arange(len(measured))\\n","    fig, axes = plt.subplots(1, 3, figsize=(7.2, 3.0), constrained_layout=True)\\n","    if measured[\'fit_seconds\'].notna().any():\\n","        axes[0].bar(x, pd.to_numeric(measured[\'fit_seconds\'],errors=\'coerce\') / 3600, color=COLORS[\'blue\'])\\n","    else:\\n","        axes[0].text(.5,.5,\'NOT_MEASURED\',ha=\'center\',va=\'center\',transform=axes[0].transAxes)\\n","    axes[0].set(ylabel=\'Coach.fit wall time (h)\', title=\'Training + native evaluation / I/O\')\\n","    axes[1].bar(x, measured[\'preprocessing_seconds\'], color=COLORS[\'green\'])\\n","    axes[1].set(ylabel=\'Preprocessing wall time (s)\', title=\'Total model preprocessing\')\\n","    axes[2].bar(x - .18, measured[\'peak_training_allocated_mib\'], width=.36, color=COLORS[\'blue\'], label=\'Allocated\')\\n","    axes[2].bar(x + .18, measured[\'peak_training_reserved_mib\'], width=.36, color=COLORS[\'orange\'], label=\'Reserved\')\\n","    axes[2].set(ylabel=\'Training allocator peak (MiB)\', title=\'Training-epoch CUDA memory\')\\n","    axes[2].legend(frameon=False)\\n","    for ax in axes:\\n","        ax.set_xticks(x, labels)\\n","        ax.grid(axis=\'y\', alpha=.2)\\n","    fig.suptitle(f\'ECLIPSE w/o CSE resource profile | seed {SEED}\', fontsize=11)\\n","    save_figure(fig, f\'{STEM}_resource_profile\',\\n","                \'ECLIPSE w/o CSE resource measurements, not a comparison against another model. \'\\n","                \'Coach.fit includes native evaluation and I/O; preprocessing is separate. \'\\n","                \'Memory excludes evaluation peaks. Cache state and runtime remain in manifests.\',\\n","                [RUN_ROOT / \'tables\' / f\'{STEM}_selected_test_overview.csv\'])\\n","\\n","resource_fields=[\'preprocessing_seconds\',\'fit_seconds\',\'train_epoch_seconds_sum\',\'subprocess_wall_seconds\',\'peak_training_allocated_mib\',\'peak_training_reserved_mib\',\'memory_scope\',\'preprocessing_cache_hit\']\\n","pd.DataFrame([{k:RESULTS[\'electronics\'].get(k) for k in resource_fields}]).to_csv(RUN_ROOT/\'tables\'/f\'{STEM}_resource_summary.csv\',index=False,na_rep=\'NOT_MEASURED\')\\n"],"execution_count":null,"outputs":[]},{"cell_type":"markdown","id":"6a4615cf","metadata":{},"source":["## 13. Complete archive — native files + external log + source/config + figures/tables\\n","ZIP is outside the run folder and uses a unique session ID. The inventory records bytes and SHA256; an external sidecar records archive bytes/SHA256. Read-only recovery copies existing artifacts into a new export folder without changing the original. Native checkpoint bytes are packaged but never loaded by reporting cells.\\n","\\n","A nonrecursive source template includes every logical cell; it excludes only its own embedding assignment and runtime outputs. For the exact submitted Kaggle notebook, also set NOTEBOOK_SOURCE_PATH or download the executed notebook separately. Human cross-read remains pending.\\n"]},{"cell_type":"code","id":"d794eacd","metadata":{},"source":["import zipfile\\n","write_json(RUN_ROOT/\'provenance\'/\'notebook_source_template.ipynb\',NOTEBOOK_SOURCE_TEMPLATE)\\n","script=\'\\\\n\\\\n\'.join(\'# CELL \'+str(i)+\'\\\\n\'+\'\'.join(c[\'source\']) for i,c in enumerate(NOTEBOOK_SOURCE_TEMPLATE[\'cells\']) if c[\'cell_type\']==\'code\')\\n","(RUN_ROOT/\'helpers\'/\'orchestration_source.py\').write_text(script,encoding=\'utf-8\')\\n","if NOTEBOOK_SOURCE_PATH:\\n","    original=Path(NOTEBOOK_SOURCE_PATH).expanduser().resolve()\\n","    if not original.is_file():raise FileNotFoundError(original)\\n","    shutil.copy2(original,RUN_ROOT/\'provenance\'/\'submitted_notebook.ipynb\')\\n","    write_json(RUN_ROOT/\'provenance\'/\'submitted_notebook_receipt.json\',{\'path\':str(original),\'sha256\':file_sha256(original)})\\n","if RUN_MODE==\'report_only\':\\n","    native=ARTIFACT_PATHS[\'electronics\'];copied=RUN_ROOT/\'artifacts\'/STEM\\n","    shutil.copytree(native,copied,dirs_exist_ok=False)\\n","    log=LOG_PATHS[\'electronics\']\\n","    if log and log.is_file():shutil.copy2(log,RUN_ROOT/\'logs\'/f\'{STEM}.log\')\\n","    m=json.loads((native/\'manifest.json\').read_text(encoding=\'utf-8\'))\\n","    write_json(RUN_ROOT/\'provenance\'/\'report_only_origin.json\',{\'artifact\':str(native),\'copied_to\':str(copied),\\n","      \'stdout_log\':str(log) if log else None,\'raw_artifacts_modified\':False,\'recovered_from_native_manifest\':True,\\n","      \'original_yaml_available\':False,\'original_dirty_diff_available\':False,\'note\':\'Do not invent unavailable historical provenance.\'})\\n","    write_json(RUN_ROOT/\'configs\'/f\'{STEM}_recorded_native_config.json\',{\'v8_config\':m.get(\'v8_config\'),\'icaart_config\':m.get(\'icaart_config\'),\\n","       \'checkpoint_config\':m.get(\'checkpoint_contract\',{}).get(\'config\'),\'argv\':m.get(\'argv\')})\\n","    write_json(RUN_ROOT/\'provenance\'/f\'{STEM}_recorded_native_environment.json\',{k:m.get(k) for k in (\'python\',\'torch\',\'freerec\',\'gpu\',\'commit\',\'precision\')})\\n","for kind in (\'learning_curves\',\'training_memory\',\'calibration_diagnostics\'):\\n","    for extension in (\'pdf\',\'png\'):\\n","        path=RUN_ROOT/\'figures\'/f\'{STEM}_{kind}.{extension}\'\\n","        if not path.is_file():raise FileNotFoundError(f\'Missing required figure {path}; inspect actual diagnostics, do not fabricate values.\')\\n","entries=json.loads((RUN_ROOT/\'figures\'/\'figure_manifest.json\').read_text(encoding=\'utf-8\'))\\n","caption_text=\'# ECLIPSE w/o CSE figure captions\\\\n\\\\n\'\\n","for entry in entries:\\n","    caption_text+=\'## \'+entry[\'stem\']+\'\\\\n\\\\n\'+entry[\'caption\']+\'\\\\n\\\\nSources:\\\\n\'+\'\\\\n\'.join(\'- \'+p for p in entry[\'sources\'])+\'\\\\n\\\\n\'\\n","(RUN_ROOT/\'figures\'/\'FIGURE_CAPTIONS.md\').write_text(caption_text,encoding=\'utf-8\')\\n","run=json.loads((RUN_ROOT/\'run_manifest.json\').read_text(encoding=\'utf-8\'))\\n","run.update(status=\'COMPLETED_PENDING_HUMAN_REVIEW\',selected_epoch=RESULTS[\'electronics\'][\'selected_epoch\'],actual_n=1,\\n","      comparability=RESULTS[\'electronics\'][\'comparability\'][\'status\'],reviewed_by=None,matched_full_comparison=\'NOT_COMPUTED\')\\n","write_json(RUN_ROOT/\'run_manifest.json\',run)\\n","(RUN_ROOT/\'README.md\').write_text(f\\"\\"\\"# ECLIPSE w/o CSE | Electronics | seed {SEED}\\n","Mode: {RUN_MODE}; unique attempt/export: {SESSION_ID}\\n","Protocol: 64D, 500 epochs; eta0, PEEC delta1, LC .01. Native VALID-selected TEST only.\\n","Pinned algorithm commit: {RUNTIME_COMMIT}; historical source differences require separate review.\\n","Primary table: tables/{STEM}_selected_test.csv (canonical uppercase metrics).\\n","Native receipt/checkpoint/telemetry: artifacts/{STEM}/\\n","External stdout: logs/; configs/ and provenance/ retain configuration/source/environment evidence.\\n","Figures: PDF vector + PNG300dpi, captions and source CSV/JSON. One measured run only; no seed SD/p-value.\\n","Timing: preprocessing and Coach.fit separate; subprocess wall is not Coach.fit; missing data NOT_MEASURED.\\n","Memory: child training-epoch CUDA allocator, excludes evaluation/driver/other GPU processes.\\n","Source template includes logical cells without self-embedding. Exact submitted notebook copied only if supplied.\\n","Human review remains pending. Reporting does not load checkpoint tensors or reproduce model metrics.\\n","\\"\\"\\",encoding=\'utf-8\')\\n","inventory=[]\\n","for path in sorted(RUN_ROOT.rglob(\'*\')):\\n","    if path.is_file() and path.name!=\'file_inventory.json\':\\n","        inventory.append({\'path\':path.relative_to(RUN_ROOT).as_posix(),\'bytes\':path.stat().st_size,\'sha256\':file_sha256(path)})\\n","write_json(RUN_ROOT/\'file_inventory.json\',inventory)\\n","archive=RUN_ROOT.parent/f\'{RUN_ROOT.name}.zip\'\\n","with zipfile.ZipFile(archive,\'x\',compression=zipfile.ZIP_DEFLATED,compresslevel=1) as z:\\n","    for path in sorted(RUN_ROOT.rglob(\'*\')):\\n","        if path.is_file():z.write(path,arcname=str(Path(RUN_ROOT.name)/path.relative_to(RUN_ROOT)))\\n","with zipfile.ZipFile(archive) as z:\\n","    if z.testzip() is not None:raise RuntimeError(\'Archive CRC validation failed\')\\n","receipt={\'archive\':str(archive),\'bytes\':archive.stat().st_size,\'sha256\':file_sha256(archive),\'file_count\':len(inventory)+1,\\n","         \'status\':\'PACKAGED_PENDING_HUMAN_REVIEW\',\'run_root\':str(RUN_ROOT)}\\n","write_json(archive.with_suffix(\'.archive_receipt.json\'),receipt)\\n","print(json.dumps(receipt,indent=2))\\n","try:\\n","    from IPython.display import FileLink\\n","    display(FileLink(str(archive)));display(FileLink(str(archive.with_suffix(\'.archive_receipt.json\'))))\\n","except ImportError:pass\\n"],"execution_count":null,"outputs":[]}],"metadata":{"kernelspec":{"display_name":"Python 3","language":"python","name":"python3"},"language_info":{"name":"python","file_extension":".py"},"eclipse":{"arm":"wo_cse","dataset":"electronics","seed":3,"runtime_commit":"5212263e87c5c1cfe6e48c91afa90201a48f5929","review_date":"2026-10-19"}},"nbformat":4,"nbformat_minor":5}')


## 3. Pinned source and environment — train mode only; no in-place patches or CPU fallback


In [ ]:
if RUN_MODE == 'train':
    import importlib.metadata
    import importlib.util
    import subprocess

    if REPO_OVERRIDE is not None:
        REPO_ROOT=Path(REPO_OVERRIDE).expanduser().resolve()
    else:
        REPO_ROOT=WORK_DIR/'eclipse_sources'/RUNTIME_COMMIT
        if not REPO_ROOT.exists():
            REPO_ROOT.mkdir(parents=True,exist_ok=False)
            subprocess.run(['git','init',str(REPO_ROOT)],check=True)
            subprocess.run(['git','-C',str(REPO_ROOT),'remote','add','origin',REPO_URL],check=True)
            subprocess.run(['git','-C',str(REPO_ROOT),'fetch','--depth','1','origin',RUNTIME_COMMIT],check=True)
            subprocess.run(['git','-C',str(REPO_ROOT),'checkout','--detach',RUNTIME_COMMIT],check=True)
    actual=subprocess.run(['git','-C',str(REPO_ROOT),'rev-parse','HEAD'],text=True,capture_output=True,check=True).stdout.strip()
    if actual!=RUNTIME_COMMIT:raise RuntimeError('Use the pinned checkout; existing source is never reset.')
    for args,name in ((['status','--porcelain'],'git_status.txt'),(['diff','HEAD','--'],'source_diff.patch'),(['remote','-v'],'git_remotes.txt')):
        result=subprocess.run(['git','-C',str(REPO_ROOT),*args],text=True,encoding='utf-8',capture_output=True,check=True)
        (RUN_ROOT/'provenance'/name).write_text(result.stdout,encoding='utf-8')
    if not (REPO_ROOT / 'main_stair_icaart.py').is_file():
        raise FileNotFoundError(REPO_ROOT / 'main_stair_icaart.py')

    ENV = os.environ.copy()
    ENV['PYTHONPATH'] = os.pathsep.join(filter(None, [str(REPO_ROOT), ENV.get('PYTHONPATH', '')]))
    ENV['PYTHONUTF8'] = '1'
    ENV['PYTHONIOENCODING'] = 'utf-8'
    ENV['PYTHONHASHSEED'] = str(SEED)
    ENV['CUDA_VISIBLE_DEVICES'] = str(GPU_INDEX)
    ENV.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')

    # Do not import Torch into the notebook kernel: all CUDA work belongs to the training subprocess.
    if INSTALL_MISSING_DEPENDENCIES:
        packages = {
            'freerec': ('freerec==0.9.7', True),
            'torch_geometric': ('torch-geometric==2.8.1', True),
            'numpy': ('numpy==2.1.3', True), 'scipy': ('scipy==1.16.3', True),
            'yaml': ('PyYAML==6.0.3', True), 'pandas': ('pandas==2.3.3', True),
            'matplotlib': ('matplotlib==3.10.0', True), 'prettytable': ('prettytable==3.18.0', True),
            'sklearn': ('scikit-learn==1.6.1', True), 'psutil': ('psutil==5.9.5', True),
        }
        for module, (requirement, no_deps) in packages.items():
            missing = importlib.util.find_spec(module) is None
            if module == 'freerec' and not missing:
                missing = importlib.metadata.version('freerec') != '0.9.7'
            if missing:
                command = [sys.executable, '-m', 'pip', 'install', '--quiet']
                if no_deps:
                    command.append('--no-deps')
                subprocess.run(command + [requirement], check=True)

    expected_sources = SEED1_REFERENCE['electronics']['contract']['sources']
    source_report = {}
    for name, expected in expected_sources.items():
        choices = [REPO_ROOT / name, REPO_ROOT / 'models' / name, REPO_ROOT / 'optimizers' / name]
        path = next((p for p in choices if p.is_file()), None)
        if path is None:
            raise FileNotFoundError(f'Missing seed-1 source: {name}')
        payload = path.read_bytes()
        actual = hashlib.sha256(payload).hexdigest()
        lf_digest = hashlib.sha256(payload.replace(b'\r\n', b'\n')).hexdigest()
        source_report[name] = {'path': str(path.relative_to(REPO_ROOT)), 'sha256': actual,
                               'line_endings_only': actual != expected and lf_digest == expected}
        if expected not in (actual, lf_digest):
            raise RuntimeError(f'Algorithm source differs from seed 1: {path}. Use the pinned runtime commit.')
    write_json(RUN_ROOT / 'provenance' / 'source_check.json', source_report)

    probe = r"""
    import json, platform, torch
    import models.freerec_compat
    import freerec
    from main_stair_icaart import build_config
    from stair_icaart_reporting import validate_selected_receipt
    print('ECLIPSE_ENV=' + json.dumps({'python': platform.python_version(), 'torch': str(torch.__version__),
     'freerec': str(freerec.__version__), 'torch_geometric': __import__('importlib.metadata',fromlist=['version']).version('torch-geometric'), 'cuda': torch.cuda.is_available(),
     'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}))
    """
    probe = __import__('textwrap').dedent(probe)
    checked = subprocess.run([sys.executable, '-c', probe], cwd=REPO_ROOT, env=ENV,
                             text=True, encoding='utf-8', capture_output=True)
    if checked.stdout:
        print(checked.stdout)
    if checked.stderr:
        print(checked.stderr)
    checked.check_returncode()
    line = next(line for line in checked.stdout.splitlines() if line.startswith('ECLIPSE_ENV='))
    RUNTIME_ENV = json.loads(line.split('=', 1)[1])
    if not RUNTIME_ENV['cuda']:
        raise RuntimeError('Enable a Kaggle GPU accelerator before training.')
    reference = SEED1_REFERENCE['electronics']
    deviations = {key: {'reference': reference[key], 'current': RUNTIME_ENV[key]}
                  for key in ('torch', 'freerec','gpu') if RUNTIME_ENV[key] != reference[key]}
    if RUNTIME_ENV['python'].split('.')[:2] != reference['python'].split('.')[:2]:
        deviations['python'] = {'reference': reference['python'], 'current': RUNTIME_ENV['python']}
    if RUNTIME_ENV.get('torch_geometric') != '2.8.1':
        deviations['torch_geometric'] = {'reference':'2.8.1','current':RUNTIME_ENV.get('torch_geometric')}
    write_json(RUN_ROOT / 'provenance' / 'environment.json', {'runtime': RUNTIME_ENV, 'deviations': deviations,
               'strict_runtime': STRICT_RUNTIME, 'gpu_index': GPU_INDEX})
    if deviations and STRICT_RUNTIME:
        raise RuntimeError(f'Runtime differs from seed 1: {deviations}. Restore the seed-1 environment; '
                           'setting STRICT_RUNTIME=False explicitly records a protocol deviation.')
    if deviations:
        print('PROTOCOL DEVIATION:', deviations)
    if RUNTIME_ENV['gpu'] != reference['gpu']:
        print('HARDWARE DEVIATION:', RUNTIME_ENV['gpu'], 'vs', reference['gpu'],
              '- runtime measurements are not a matched-hardware comparison.')
    freeze = subprocess.run([sys.executable, '-m', 'pip', 'freeze'], text=True, capture_output=True, check=True)
    (RUN_ROOT / 'provenance' / 'pip_freeze.txt').write_text(freeze.stdout, encoding='utf-8')
    commit = subprocess.run(['git', '-C', str(REPO_ROOT), 'rev-parse', 'HEAD'], text=True, capture_output=True)
    ACTUAL_COMMIT = commit.stdout.strip() if commit.returncode == 0 else 'unavailable'
    print('Runtime ready:', REPO_ROOT, '| commit:', ACTUAL_COMMIT)
    print('Import/environment checks only; no model test or training has been executed.')

    driver=subprocess.run(['nvidia-smi','--query-gpu=name,driver_version,memory.total','--format=csv'],text=True,capture_output=True)
    (RUN_ROOT/'provenance'/'nvidia_smi.txt').write_text(driver.stdout+driver.stderr,encoding='utf-8')

else:
    print('report_only: skipped training setup; no model/network/GPU invocation.')


## 4. Input discovery and split/feature hashes — attach data before Kaggle batch; report-only skips data


In [ ]:
if RUN_MODE == 'train':
    import shutil

    REQUIRED_FILES = ('train.txt', 'valid.txt', 'test.txt', 'textual_modality.pkl', 'visual_modality.pkl')


    def dataset_search_roots(extra_roots=()):
        # Include writable bridges and caches created by earlier interactive sessions.
        return [REPO_ROOT / 'data', DATA_ROOT, WORK_DIR / 'data',
                Path('/kaggle/data'), Path('/kaggle/input'),
                Path.home() / '.cache' / 'kagglehub' / 'datasets',
                *map(Path, DATASET_SEARCH_ROOTS), *map(Path, extra_roots)]


    def discover_datasets(extra_roots=()):
        found = {}
        unknown = set(DATASET_SOURCE_OVERRIDES) - set(DATASET_FOLDERS)
        if unknown:
            raise ValueError(f'Unknown DATASET_SOURCE_OVERRIDES keys: {sorted(unknown)}')
        for key in SELECTED_DATASETS:
            if key not in DATASET_SOURCE_OVERRIDES:
                continue
            path = Path(DATASET_SOURCE_OVERRIDES[key]).expanduser()
            absent = [name for name in REQUIRED_FILES if not (path / name).is_file()]
            if absent:
                raise FileNotFoundError(f'Incomplete explicit dataset source for {key}: {path}; missing {absent}')
            found[key] = path
        aliases = {'baby': ('baby',), 'sports': ('sport',), 'electronics': ('electronic',)}
        visited = set()
        for root in dataset_search_roots(extra_roots):
            root = root.expanduser()
            if not root.is_dir() or str(root.resolve()) in visited:
                continue
            visited.add(str(root.resolve()))
            for current, directories, files in os.walk(root, followlinks=False):
                directories[:] = sorted(d for d in directories if d not in
                                        {'.git', 'chunks', '__pycache__', '.stair_icaart_graph_cache'})
                if not set(REQUIRED_FILES).issubset(files):
                    continue
                path = Path(current)
                for key in SELECTED_DATASETS:
                    if any(alias in path.name.lower() for alias in aliases[key]):
                        found.setdefault(key, path)
        return found


    def missing_dataset_message(missing):
        return (
            f'Missing processed datasets: {sorted(missing)}.\n'
            f'On Kaggle, attach {DATASET_HANDLE!r} with Add Input BEFORE Save & Run All '
            '(or attach an equivalent processed dataset matching the seed-1 fingerprints). '
            'Save a new notebook version after attaching it. Internet access alone does not '
            'attach a datasource, and kagglehub cannot attach a new one in a non-interactive run.\n'
            'Alternatively set DATASET_SOURCE_OVERRIDES to the existing per-dataset folders, '
            'or DATASET_SEARCH_ROOTS to their parent directory. Each dataset folder must contain: '
            + ', '.join(REQUIRED_FILES) + '.\n'
            'Searched roots: ' + ', '.join(str(p.expanduser()) for p in dataset_search_roots())
            + f'\nDiscovery receipt: {RUN_ROOT / "provenance" / "data_discovery.json"}'
        )


    def resolve_dataset_sources():
        sources = discover_datasets()
        missing = set(SELECTED_DATASETS) - sources.keys()
        on_kaggle = Path('/kaggle/working').is_dir()
        discovery = {'dataset_handle': DATASET_HANDLE, 'selected_datasets': list(SELECTED_DATASETS),
                     'searched_roots': [str(p.expanduser()) for p in dataset_search_roots()],
                     'on_kaggle': on_kaggle, 'auto_download_requested': AUTO_DOWNLOAD_DATA,
                     'download_attempted': False}

        def record(status):
            discovery.update(status=status, missing=sorted(missing),
                             sources={key: str(path.resolve()) for key, path in sources.items()})
            write_json(RUN_ROOT / 'provenance' / 'data_discovery.json', discovery)

        if missing and on_kaggle:
            # Never try to attach a datasource during a Kaggle batch job. The caller
            # must attach inputs before execution; this also makes batch runs repeatable.
            record('MISSING_ATTACHED_INPUT')
            raise FileNotFoundError(missing_dataset_message(missing))
        if missing and AUTO_DOWNLOAD_DATA:
            discovery['download_attempted'] = True
            try:
                import kagglehub
                downloaded = kagglehub.dataset_download(DATASET_HANDLE)
                discovery['downloaded_root'] = str(downloaded)
                sources.update(discover_datasets([downloaded]))
                missing = set(SELECTED_DATASETS) - sources.keys()
            except Exception as exc:
                discovery['download_error'] = {'type': type(exc).__name__, 'message': str(exc)}
                record('DOWNLOAD_FAILED')
                raise FileNotFoundError(missing_dataset_message(missing)) from exc
        if missing:
            record('MISSING_PROCESSED_DATA')
            raise FileNotFoundError(missing_dataset_message(missing))
        record('FOUND_PENDING_FINGERPRINT_CHECK')
        return sources


    sources = resolve_dataset_sources()

    DATA_RECEIPTS = {}
    for key in SELECTED_DATASETS:
        source = sources[key].resolve()
        reference = SEED1_REFERENCE[key]
        expected = dict(reference['contract']['splits'])
        expected.update({record['file']: record['sha256'] for record in reference['features']})
        destination = DATA_ROOT / 'Processed' / DATASET_FOLDERS[key]
        if destination.is_symlink():
            raise RuntimeError(f'Processed destination must be a writable directory, not a symlink: {destination}')
        destination.mkdir(parents=True, exist_ok=True)
        for name, expected_hash in expected.items():
            original = source / name
            if not original.is_file():
                raise FileNotFoundError(original)
            actual_hash = file_sha256(original)
            if actual_hash != expected_hash:
                raise RuntimeError(f'Dataset differs from seed 1: {original}')
            target = destination / name
            if target.is_symlink() and not target.exists():
                raise FileExistsError(f'Broken input link preserved; choose a fresh DATA_ROOT: {target}')
            if target.exists():
                if file_sha256(target) != expected_hash:
                    raise RuntimeError(f'Existing linked/copied input has a different fingerprint: {target}')
            else:
                try:
                    target.symlink_to(original)
                except OSError:
                    shutil.copy2(original, target)
        if (destination / 'chunks').is_symlink():
            raise RuntimeError(f'FreeRec chunks must be writable, not an input symlink: {destination / "chunks"}')
        (destination / 'chunks').mkdir(exist_ok=True)
        DATA_RECEIPTS[key] = {'source': str(source), 'processed': str(destination), 'fingerprints': expected}
        print(f'{DATASET_TITLES[key]}: split and feature fingerprints match seed 1.')
    write_json(RUN_ROOT / 'provenance' / 'data_check.json', DATA_RECEIPTS)

else:
    print('report_only: skipped training setup; no model/network/GPU invocation.')


## 5. YAML and resolved CLI preflight — only eta changes; no model constructed


In [ ]:
if RUN_MODE == 'train':
    import yaml

    CONFIG_PATHS = {}
    ARTIFACT_PATHS = {}
    LOG_PATHS = {}
    for key in SELECTED_DATASETS:
        title = key.capitalize()
        original = REPO_ROOT / 'configs' / f'Amazon2014{title}_STAIR_PEEC.yaml'
        if not original.is_file():
            raise FileNotFoundError(original)
        config = yaml.safe_load(original.read_text(encoding='utf-8'))
        config.update(SEED1_REFERENCE[key]['v8_config'])
        config.update(SEED1_REFERENCE[key]['icaart_config'])
        config['eta'] = 0.0
        config['evidence_delta'] = 1.0
        config.pop('evidence_seed', None)  # Inactive; do not confuse the evidence partition with training RNG.
        config.update(seed=SEED, root=str(DATA_ROOT), dataset=DATASET_FOLDERS[key],
                      num_workers=SEED1_REFERENCE[key]['contract']['config']['num_workers'],
                      eval_valid=True, eval_test=False, ranking='full', which4best='NDCG@20',
                      resume=False, resume_from='')
        config['mfiles'] = ','.join(config['mfiles'])
        config['num_neighbors'] = '-'.join(map(str, config['num_neighbors']))
        artifact = RUN_ROOT / 'artifacts' / STEM
        config['artifact_dir'] = str(artifact)
        config['graph_cache_dir'] = str(GRAPH_CACHE_ROOT / key)
        target = RUN_ROOT / 'configs' / f'{STEM}.yaml'
        target.write_text(yaml.safe_dump(config, sort_keys=False), encoding='utf-8')
        shutil.copy2(original,RUN_ROOT/"configs"/f"{STEM}_original.yaml")
        CONFIG_PATHS[key], ARTIFACT_PATHS[key] = target, artifact
        LOG_PATHS[key] = RUN_ROOT / 'logs' / f'{STEM}.log'


    def build_command(key):
        if key not in SELECTED_DATASETS:
            raise ValueError(f'{key} is not selected; edit SELECTED_DATASETS and rerun setup.')
        return [sys.executable, '-u', str(REPO_ROOT / 'main_stair_icaart.py'),
                '--config', str(CONFIG_PATHS[key]), '--seed', str(SEED),
                '--root', str(DATA_ROOT), '--dataset', DATASET_FOLDERS[key],
                '--device', '0', '--ranking', 'full',
                '--artifact-dir', str(ARTIFACT_PATHS[key]),
                '--graph-cache-dir', str(GRAPH_CACHE_ROOT / key)]


    config_probe = r"""
    import json, sys
    from main_stair_icaart import build_config
    cfg = build_config(sys.argv[1:])
    keys = json.loads(__import__('os').environ['ECLIPSE_CHECK_KEYS'])
    receipt = {key: getattr(cfg, key) for key in keys}
    print('ECLIPSE_CONFIG=' + json.dumps(receipt))
    """
    config_probe = __import__('textwrap').dedent(config_probe)
    for key in SELECTED_DATASETS:
        expected = dict(SEED1_REFERENCE[key]['v8_config'])
        expected.update(SEED1_REFERENCE[key]['icaart_config'])
        expected['eta'] = 0.0
        expected['evidence_delta'] = 1.0
        expected.update(seed=SEED, dataset=DATASET_FOLDERS[key], ranking='full', which4best='NDCG@20',
                        eval_valid=True, eval_test=False, resume=False, resume_from='',
                        num_workers=SEED1_REFERENCE[key]['contract']['config']['num_workers'])
        probe_env = ENV.copy()
        probe_env['ECLIPSE_CHECK_KEYS'] = json.dumps(list(expected))
        result = subprocess.run([sys.executable, '-c', config_probe, *build_command(key)[3:]],
                                cwd=REPO_ROOT, env=probe_env, text=True, encoding='utf-8', capture_output=True)
        if result.returncode:
            print(result.stdout, result.stderr)
            result.check_returncode()
        payload = next(line.split('=', 1)[1] for line in result.stdout.splitlines()
                       if line.startswith('ECLIPSE_CONFIG='))
        resolved = json.loads(payload)
        changes = {name: {'expected': value, 'actual': resolved.get(name)}
                   for name, value in expected.items() if resolved.get(name) != value}
        if changes:
            raise RuntimeError(f'{key}: resolved CLI/YAML protocol mismatch: {changes}')
        write_json(RUN_ROOT / 'provenance' / f'{key}_resolved_config.json', resolved)
        print(f'{key}: seed={resolved["seed"]}, epochs={resolved["epochs"]}, '
              f'batch={resolved["batch_size"]}, gamma={resolved["gamma"]}, '
              f'weight_decay={resolved["weight_decay"]}, cl_chunk={resolved["cl_chunk_size"]}')
    write_json(RUN_ROOT / 'run_manifest.json', {
        'method': 'ECLIPSE', 'arm': ARM, 'arm_id': ARM, 'status':'READY', 'training_seed': SEED,
        'datasets': list(SELECTED_DATASETS), 'session_id': SESSION_ID,
        'reference_commit': RUNTIME_COMMIT, 'actual_commit': ACTUAL_COMMIT,
        'runtime': RUNTIME_ENV, 'runtime_deviations': deviations,
        'selection': 'validation NDCG@20', 'test_policy': 'report only VALID-selected TEST; native terminal TEST may also be evaluated',
        'configs': {k: {'path': str(p), 'sha256': file_sha256(p)} for k, p in CONFIG_PATHS.items()},
        'dataset_paths': DATA_RECEIPTS,
        'note': 'w/o CSE only: eta=0; PEEC delta1 and LC .01 retained; no resume.',
    })

else:
    print('report_only: skipped training setup; no model/network/GPU invocation.')


## 6. Artifact validators — semantic/evidence invariants, not Full final-graph equality
Checks eta0 in effective metadata, PEEC/LC retained, input/source contracts and native selected TEST. Historical source/runtime differences are recorded for human review. Diagnostics use an uncalibrated **semantic-only eta0** reference.


In [ ]:
import math
import re
import signal
import time
import pandas as pd
import numpy as np
from IPython.display import display

RESULTS = {}


def read_jsonl(path):
    path = Path(path)
    if not path.is_file():
        return []
    result = []
    with path.open(encoding='utf-8') as handle:
        for number, line in enumerate(handle, 1):
            if line.strip():
                try:
                    result.append(json.loads(line))
                except json.JSONDecodeError as exc:
                    raise ValueError(f'Malformed JSONL: {path}, line {number}') from exc
    return result


def compare_manifest(key, manifest, expected_seed):
    ref=SEED1_REFERENCE[key];problems={}
    if manifest.get('seed')!=expected_seed:problems['seed']=manifest.get('seed')
    if manifest.get('selection')!='validation NDCG@20':problems['selection']=manifest.get('selection')
    for name,expected in dict(ref['v8_config'],eta=0.0).items():
        if manifest.get('v8_config',{}).get(name)!=expected:problems['v8_config.'+name]=manifest.get('v8_config',{}).get(name)
    if manifest.get('icaart_config')!=ref['icaart_config']:problems['evidence_config']=manifest.get('icaart_config')
    contract=manifest.get('checkpoint_contract',{})
    for name in ('splits','candidate_hashes','data_fingerprint'):
        if contract.get(name)!=ref['contract'][name]:problems[name]=contract.get(name)
    for name,expected in ref['contract']['config'].items():
        if name=='seed':expected=expected_seed
        if name=='eta':expected=0.0
        if contract.get('config',{}).get(name)!=expected:problems['config.'+name]=contract.get('config',{}).get(name)
    if manifest.get('features')!=ref['features']:problems['features']=manifest.get('features')
    if manifest.get('precision')!={'tf32':False,'amp':False}:problems['precision']=manifest.get('precision')
    e=manifest.get('icaart_manifest',{});g=manifest.get('graph',{});d=g.get('icaart',{})
    for name,expected in ref['evidence_invariants'].items():
        if e.get(name)!=expected:problems['evidence.'+name]=e.get(name)
    if e.get('training_seed')!=expected_seed:problems['evidence.training_seed']=e.get('training_seed')
    for name,actual in (('graph.eta',g.get('eta')),('graph.effective_eta',g.get('effective_eta')),('icaart.effective_eta',d.get('effective_eta'))):
        if actual!=0.0:problems[name]=actual
    if e.get('null_scope')!='full_activity' or e.get('normalization')!='actual_degree':problems['null_normalization']=e
    fp=contract.get('graph_fingerprint')
    if not isinstance(fp,str) or len(fp)!=64:problems['final_graph_fingerprint']=fp
    for name,actual in (('graph',g.get('graph_fingerprint')),('icaart_manifest',e.get('graph_fingerprint')),('diagnostics',d.get('s_ica_fingerprint'))):
        if actual!=fp:problems['final_graph_consistency.'+name]=actual
    if problems:raise RuntimeError(f'{key}: w/o CSE comparability failed: {problems}')
    source_diff={k:{'reference':v,'actual':contract.get('sources',{}).get(k)} for k,v in ref['contract']['sources'].items() if contract.get('sources',{}).get(k)!=v}
    runtime_diff={k:{'reference':ref[k],'actual':manifest.get(k)} for k in ('torch','freerec','gpu') if manifest.get(k)!=ref[k]}
    if str(manifest.get('python','')).split('.')[:2]!=ref['python'].split('.')[:2]:runtime_diff['python']=manifest.get('python')
    if RUN_MODE=='train' and source_diff:raise RuntimeError(f'Source mismatch: {source_diff}')
    if RUN_MODE=='train' and STRICT_RUNTIME and runtime_diff:raise RuntimeError(f'Runtime mismatch: {runtime_diff}')
    return {'status':'MATCH_WITH_REVIEW_REQUIRED' if source_diff or runtime_diff else 'MATCH',
        'arm':ARM,'training_seed':expected_seed,'model_intervention':{'eta':[0.1,0.0]},
        'same_candidates_semantic_evidence':True,'same_final_graph_as_full':'NOT_REQUIRED',
        'dose_reference':'uncalibrated semantic-only eta0','source_deviations':source_diff,'runtime_deviations':runtime_diff,
        'operator_check_scope':'manifest and pinned source contract; not an independent sparse-tensor reconstruction',
        'cf_preprocessing_may_remain':True}


def validate_history(telemetry, evaluations, receipt):
    if [row.get('epoch') for row in telemetry]!=list(range(1,501)):raise ValueError('Require exactly ordered epochs1..500; no duplicate/missing epochs or implicit resume')
    for row in telemetry:
        for name in ('loss','bpr_loss','nlgcl_loss','weighted_nlgcl_loss','seconds','peak_allocated_gib','peak_reserved_gib'):
            value=row.get(name)
            if isinstance(value,bool) or not isinstance(value,(int,float)) or not math.isfinite(value) or value<0:raise ValueError(f'Invalid {name} at epoch {row["epoch"]}')
        if row.get('lambda_nlgcl')!=.01:raise ValueError('LC must stay enabled')
        if not math.isclose(row['weighted_nlgcl_loss'],.01*row['nlgcl_loss'],rel_tol=1e-6,abs_tol=1e-7):raise ValueError('Weighted LC mismatch')
        if not math.isclose(row['loss'],row['bpr_loss']+row['weighted_nlgcl_loss'],rel_tol=1e-6,abs_tol=1e-7):raise ValueError('Objective mismatch')
    valid=[r for r in evaluations if r.get('mode')=='valid' and not r.get('selected_checkpoint')]
    if not valid:raise ValueError('Missing VALID history')
    seen=set()
    for row in valid:
        epoch=row.get('epoch');v=row.get('metrics',{}).get('NDCG@20')
        if isinstance(epoch,bool) or not isinstance(epoch,int) or not 0<=epoch<=500 or epoch in seen:raise ValueError('Invalid/duplicate VALID epoch')
        seen.add(epoch)
        if isinstance(v,bool) or not isinstance(v,(int,float)) or not math.isfinite(v):raise ValueError('Invalid VALID NDCG@20')
    maximum=max(r['metrics']['NDCG@20'] for r in valid)
    tied=[r['epoch'] for r in valid if r['metrics']['NDCG@20']==maximum]
    if receipt['epoch'] not in tied:raise ValueError('Selected epoch is not a VALID maximum')
    selected=[r for r in evaluations if r.get('mode')=='test' and r.get('selected_checkpoint') is True]
    if len(selected)!=1 or selected[0]!=receipt:raise ValueError('Selected receipt/history mismatch')
    return {'valid_max_epochs':tied,'tie_rule':'native receipt retained; inspect native log if tied' if len(tied)>1 else 'unique maximum',
        'test_events':[{'epoch':r.get('epoch'),'selected':r.get('selected_checkpoint')} for r in evaluations if r.get('mode')=='test']}


def validate_receipt(key, receipt, manifest, expected_seed):
    if (receipt.get('mode') != 'test' or receipt.get('selected_checkpoint') is not True
            or receipt.get('selection') != 'validation NDCG@20'
            or receipt.get('seed') != expected_seed
            or receipt.get('evidence_mode') != 'pooled_activity_excess'
            or receipt.get('evidence_delta') != 1.0):
        raise ValueError(f'{key}: not a selected-checkpoint ECLIPSE w/o CSE TEST receipt')
    if receipt.get('graph_fingerprint') != manifest.get('checkpoint_contract', {}).get('graph_fingerprint'):
        raise ValueError(f'{key}: metric receipt belongs to another operator')
    epoch = receipt.get('epoch')
    if isinstance(epoch, bool) or not isinstance(epoch, int) or not 1 <= epoch <= 500:
        raise ValueError(f'{key}: invalid selected epoch')
    normalized = receipt.get('metrics', {})
    for metric in PRIMARY_METRICS:
        value = normalized.get(metric)
        if isinstance(value, bool) or not isinstance(value, (float, int)) or not math.isfinite(value) or not 0 <= value <= 1:
            raise ValueError(f'{key}: missing/invalid measured TEST metric {metric}')
    return normalized


def export_run_tables(key):
    artifact = ARTIFACT_PATHS[key]
    manifest = json.loads((artifact / 'manifest.json').read_text(encoding='utf-8'))
    status = json.loads((artifact / 'run_status.json').read_text(encoding='utf-8'))
    if status.get('status') != 'COMPLETED' or status.get('seed') != SEED:
        raise RuntimeError(f'{key}: run is not completed for training seed {SEED}')
    comparability = compare_manifest(key, manifest, SEED)
    receipt = json.loads((artifact / 'selected_metrics.json').read_text(encoding='utf-8'))
    metrics = validate_receipt(key, receipt, manifest, SEED)
    telemetry = pd.DataFrame(read_jsonl(artifact / 'training_telemetry.jsonl'))
    evaluations = read_jsonl(artifact / 'evaluation_metrics.jsonl')
    history = validate_history(telemetry.to_dict('records'), evaluations, receipt)
    checkpoint = artifact / 'best_model.pt'
    if not checkpoint.is_file(): raise FileNotFoundError(checkpoint)
    telemetry.to_csv(RUN_ROOT / 'tables' / f'{STEM}_training_telemetry.csv', index=False, na_rep='NOT_MEASURED')
    evaluation_rows = [{k: v for k, v in record.items() if k != 'metrics'} |
                       {str(k).upper(): v for k, v in record.get('metrics', {}).items()}
                       for record in evaluations]
    pd.DataFrame(evaluation_rows).to_csv(RUN_ROOT / 'tables' / f'{STEM}_evaluations.csv',
                                       index=False, na_rep='NOT_MEASURED')
    log_text = LOG_PATHS[key].read_text(encoding='utf-8', errors='replace') if LOG_PATHS[key] and LOG_PATHS[key].is_file() else ''
    fit_matches = re.findall(r'Coach\.fit takes\s+([0-9.eE+-]+)\s+seconds', log_text)
    summary = {'method': 'ECLIPSE', 'arm': ARM, 'dataset': key, 'seed': SEED,
               'selected_epoch': receipt['epoch'], 'split': 'TEST', **metrics,
               'fit_seconds': float(fit_matches[-1]) if fit_matches else None,
               'train_epoch_seconds_sum': float(telemetry['seconds'].sum()),
               'preprocessing_seconds': manifest.get('preprocessing_seconds'),
               'peak_training_allocated_mib': float(telemetry['peak_allocated_gib'].max() * 1024),
               'peak_training_reserved_mib': float(telemetry['peak_reserved_gib'].max() * 1024),
               'memory_scope': 'child-process per-training-epoch CUDA allocator peaks; excludes evaluation peaks',
               'commit': manifest.get('commit'), 'graph_fingerprint': receipt['graph_fingerprint'],
               'artifact_dir': str(artifact), 'checkpoint_path': str(artifact / 'best_model.pt'),
               'runtime_deviations': comparability['runtime_deviations'], 'comparability': comparability}
    write_json(RUN_ROOT / 'tables' / f'{STEM}_selected_test.json', summary)
    write_json(RUN_ROOT / 'tables' / f'{STEM}_mechanism.json', manifest['graph']['icaart'])
    execution_path=RUN_ROOT/'provenance'/f'{STEM}_notebook_execution.json'
    execution=json.loads(execution_path.read_text(encoding='utf-8')) if execution_path.is_file() else {}
    summary.update(status='COMPLETED_PENDING_HUMAN_REVIEW', attempt_id=SESSION_ID,
       checkpoint_sha256=file_sha256(checkpoint), history=history,
       subprocess_wall_seconds=execution.get('subprocess_wall_seconds'),
       preprocessing_cache_hit=manifest.get('graph',{}).get('cache_hit'), reviewed_by=None)
    pd.DataFrame([{k:v for k,v in summary.items() if not isinstance(v,(dict,list))}]).to_csv(
        RUN_ROOT/'tables'/f'{STEM}_selected_test.csv',index=False,na_rep='NOT_MEASURED')
    write_json(RUN_ROOT/'tables'/f'{STEM}_selected_test.json',summary)
    write_json(RUN_ROOT/'provenance'/f'{STEM}_comparability.json',comparability)
    write_json(RUN_ROOT/'provenance'/f'{STEM}_input_receipts.json',{
       name:file_sha256(artifact/name) for name in ('manifest.json','run_status.json','selected_metrics.json','training_telemetry.jsonl','evaluation_metrics.jsonl','best_model.pt')})
    RESULTS[key] = summary
    return summary


def train_dataset(key):
    if RUN_MODE == 'report_only': return export_run_tables(key)
    if key not in SELECTED_DATASETS:
        print(f'Skipped {key}: not in SELECTED_DATASETS.')
        return None
    artifact = ARTIFACT_PATHS[key]
    status_path = artifact / 'run_status.json'
    if status_path.is_file():
        status = json.loads(status_path.read_text(encoding='utf-8'))
        if status.get('status') == 'COMPLETED':
            print(f'{key}: reading the completed attempt; no retraining.')
            return export_run_tables(key)
    # build_config() creates artifact_dir during CLI preflight, before training.
    # An empty real directory with no stdout log is not a previous run.
    log_path = LOG_PATHS[key]
    invalid_directory = artifact.is_symlink() or (artifact.exists() and not artifact.is_dir())
    has_payload = artifact.is_dir() and any(artifact.iterdir())
    has_log = log_path.exists() or log_path.is_symlink()
    if invalid_directory or has_payload or has_log:
        raise FileExistsError(f'Existing attempt payload/log preserved at {artifact}; '
                              'use report_only for completed runs, or create a new SESSION_ID for a fresh attempt. '
                              'No automatic resume/delete/retry.')
    artifact.mkdir(parents=True, exist_ok=True)
    command = build_command(key)
    write_json(RUN_ROOT / 'provenance' / f'{STEM}_command.json', {
        'argv': command, 'training_seed': SEED, 'arm_id': ARM, 'dataset': key, 'purpose':'component_ablation', 'pythonhashseed': ENV['PYTHONHASHSEED'],
        'cuda_visible_devices': ENV['CUDA_VISIBLE_DEVICES'], 'cwd': str(REPO_ROOT)})
    print('Running:', ' '.join(command))
    start = time.perf_counter()
    manifest_checked = False
    process = None
    try:
        with LOG_PATHS[key].open('x', encoding='utf-8') as log:
            process = subprocess.Popen(command, cwd=REPO_ROOT, env=ENV, stdout=subprocess.PIPE,
                                       stderr=subprocess.STDOUT, text=True, encoding='utf-8',
                                       errors='replace', bufsize=1)
            for line in process.stdout:
                log.write(line)
                log.flush()
                print(line, end='', flush=True)
                manifest_path = artifact / 'manifest.json'
                if not manifest_checked and manifest_path.is_file():
                    comparability = compare_manifest(key, json.loads(manifest_path.read_text(encoding='utf-8')), SEED)
                    write_json(RUN_ROOT / 'provenance' / f'{key}_comparability.json', comparability)
                    manifest_checked = True
            process.wait()
            process.stdout.close()
        if process.returncode != 0:
            raise RuntimeError(f'{key}: training failed with exit code {process.returncode}; see {LOG_PATHS[key]}')
        child_wall_seconds=time.perf_counter()-start
        write_json(RUN_ROOT/'provenance'/f'{STEM}_notebook_execution.json',{'status':'CHILD_COMPLETED_PENDING_EXPORT','subprocess_wall_seconds':child_wall_seconds})
        result = export_run_tables(key)
        write_json(RUN_ROOT / 'provenance' / f'{STEM}_notebook_execution.json', {
            'status': 'COMPLETED', 'subprocess_wall_seconds': child_wall_seconds,
            'exit_code': process.returncode, 'training_seed': SEED})
        print(f'\nCompleted {DATASET_TITLES[key]} | selected TEST NDCG@20={result["NDCG@20"]:.10f}')
        return result
    except BaseException as exc:
        if process is not None and process.poll() is None:
            if isinstance(exc, KeyboardInterrupt):
                process.send_signal(signal.SIGINT)
            else:
                process.terminate()
            try:
                process.wait(timeout=20)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait(timeout=10)
        write_json(RUN_ROOT / 'provenance' / f'{STEM}_notebook_execution.json', {
            'status': 'INTERRUPTED' if isinstance(exc, KeyboardInterrupt) else 'FAILED',
            'error_type': type(exc).__name__, 'error': str(exc), 'training_seed': SEED,
            'subprocess_wall_seconds': time.perf_counter() - start})
        raise


## 7. Execute one fresh run or report-only export
This is the only training invocation. Reexecution of a completed attempt exports instead of training; incomplete attempts are preserved. Interrupt/failure cleans up only the child process launched by this cell. No automatic resume/retry.


In [ ]:
result=train_dataset('electronics')
display(pd.DataFrame([{k:result[k] for k in ('arm','dataset','seed','selected_epoch',*PRIMARY_METRICS)}]))


## 8. Paper figures — PDF vector, PNG300dpi, source CSV/JSON and English captions
No synthetic curves, no TEST-based epoch selection. Resource data refer to the child process, not parent-kernel GPU measurements. Calibration stays active in w/o CSE; reference is semantic-only eta0.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 9, 'axes.titlesize': 10,
                     'axes.labelsize': 9, 'legend.fontsize': 8, 'xtick.labelsize': 8,
                     'ytick.labelsize': 8, 'lines.linewidth': 1.5,
                     'pdf.fonttype': 42, 'ps.fonttype': 42, 'savefig.dpi': 300,
                     'axes.spines.top': False, 'axes.spines.right': False})
COLORS = {'blue': '#0072B2', 'orange': '#D55E00', 'green': '#009E73',
          'purple': '#CC79A7', 'gray': '#666666'}
_existing_figure_manifest = RUN_ROOT / 'figures' / 'figure_manifest.json'
FIGURE_RECORDS = json.loads(_existing_figure_manifest.read_text(encoding='utf-8')) if _existing_figure_manifest.is_file() else []


def save_figure(fig, stem, caption, sources):
    for extension in ('pdf', 'png'):
        fig.savefig(RUN_ROOT / 'figures' / f'{stem}.{extension}', bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close(fig)
    FIGURE_RECORDS[:] = [record for record in FIGURE_RECORDS if record['stem'] != stem]
    FIGURE_RECORDS.append({'stem': stem, 'caption': caption, 'sources': list(map(str, sources))})
    write_json(RUN_ROOT / 'figures' / 'figure_manifest.json', FIGURE_RECORDS)
    (RUN_ROOT/'figures'/'FIGURE_CAPTIONS.md').write_text('# Figure captions\n\n'+'\n\n'.join('## '+r['stem']+'\n\n'+r['caption'] for r in FIGURE_RECORDS),encoding='utf-8')
    print(caption)


def completed_artifact(key):
    if key not in SELECTED_DATASETS:
        print(f'{key}: not selected; no figure generated.')
        return None
    path = ARTIFACT_PATHS[key]
    status = path / 'run_status.json'
    if not status.is_file() or json.loads(status.read_text(encoding='utf-8')).get('status') != 'COMPLETED':
        print(f'{key}: no completed run available; no figure generated.')
        return None
    return path


def plot_learning_curves(key):
    artifact = completed_artifact(key)
    if artifact is None:
        return
    telemetry = pd.DataFrame(read_jsonl(artifact / 'training_telemetry.jsonl'))
    records = [record for record in read_jsonl(artifact / 'evaluation_metrics.jsonl')
               if record.get('mode') == 'valid' and not record.get('selected_checkpoint')]
    selected = json.loads((artifact / 'selected_metrics.json').read_text(encoding='utf-8'))['epoch']
    valid = pd.DataFrame([{'epoch': record['epoch'],
                           **{str(k).upper(): v for k, v in record['metrics'].items()}}
                          for record in records]).sort_values('epoch')
    if telemetry.empty or valid.empty:
        print(f'{key}: missing measured curves; no substitute data generated.')
        return
    fig, axes = plt.subplots(1, 3, figsize=(7.2, 3.0), constrained_layout=True)
    for column, label, color in [('loss', 'Total', COLORS['blue']),
                                  ('bpr_loss', 'BPR', COLORS['orange']),
                                  ('weighted_nlgcl_loss', 'Weighted LC', COLORS['green'])]:
        if column in telemetry:
            axes[0].plot(telemetry['epoch'], telemetry[column], label=label, color=color)
    axes[0].set(xlabel='Training epoch', ylabel='Mean training loss', title='Training objective')
    axes[0].legend(frameon=False)
    for ax, metric in zip(axes[1:], ('NDCG@20', 'RECALL@20')):
        if metric in valid:
            ax.plot(valid['epoch'], valid[metric], color=COLORS['blue'], label=f'VALID {metric}')
            point = valid.loc[valid['epoch'] == selected, metric]
            if len(point):
                ax.scatter([selected], [point.iloc[-1]], color=COLORS['orange'], s=22, zorder=3)
        ax.axvline(selected, color=COLORS['gray'], linestyle='--', linewidth=1, label=f'Selected epoch {selected}')
        ax.set(xlabel='Training epoch', ylabel=f'Validation {metric}', title=f'Validation {metric}')
        ax.legend(frameon=False)
    for ax in axes:
        ax.grid(alpha=.2)
    fig.suptitle(f'ECLIPSE w/o CSE | {DATASET_TITLES[key]} | seed {SEED}', fontsize=11)
    save_figure(fig, f'{STEM}_learning_curves',
                'Training objective and validation ranking curves. The checkpoint is selected by validation NDCG@20; '
                'TEST is evaluated at the selected checkpoint and is not a learning-curve selection signal.',
                [artifact / 'training_telemetry.jsonl', artifact / 'evaluation_metrics.jsonl', artifact / 'selected_metrics.json'])


def plot_training_memory(key):
    artifact = completed_artifact(key)
    if artifact is None:
        return
    telemetry = pd.DataFrame(read_jsonl(artifact / 'training_telemetry.jsonl'))
    if telemetry.empty:
        return
    fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.8), constrained_layout=True)
    for column, label, color in [('peak_allocated_gib', 'Peak allocated', COLORS['blue']),
                                  ('peak_reserved_gib', 'Peak reserved', COLORS['orange'])]:
        if column in telemetry:
            axes[0].plot(telemetry['epoch'], telemetry[column] * 1024, label=label, color=color)
    axes[0].set(xlabel='Training epoch', ylabel='CUDA allocator memory (MiB)', title='Training-epoch allocator peaks')
    axes[0].legend(frameon=False)
    axes[1].plot(telemetry['epoch'], telemetry['seconds'], color=COLORS['green'])
    axes[1].set(xlabel='Training epoch', ylabel='Training time (s/epoch)', title='Training time per epoch')
    for ax in axes:
        ax.grid(alpha=.2)
    fig.suptitle(f'ECLIPSE w/o CSE | {DATASET_TITLES[key]} | seed {SEED}', fontsize=11)
    save_figure(fig, f'{STEM}_training_memory',
                'CUDA allocator peaks are measured in the training subprocess for each training epoch. '
                'They exclude evaluation peaks, other GPU processes and driver memory. No arbitrary hardware ceiling is drawn.',
                [artifact / 'training_telemetry.jsonl'])


def plot_calibration_diagnostics(key):
    artifact = completed_artifact(key)
    if artifact is None:
        return
    manifest = json.loads((artifact / 'manifest.json').read_text(encoding='utf-8'))
    diagnostics = manifest['graph']['icaart']
    coverage = [diagnostics.get('frac_edges_with_positive_evidence'), diagnostics.get('weighted_evidence_coverage')]
    if any(value is None for value in coverage):
        print(f'{key}: coverage was not measured.')
        return
    fig, axes = plt.subplots(1, 3, figsize=(7.2, 3.0), constrained_layout=True)
    bars = axes[0].bar(['Candidate\nedges', 'Raw semantic\nweight'], np.asarray(coverage) * 100,
                       color=[COLORS['blue'], COLORS['green']], width=.6)
    axes[0].bar_label(bars, fmt='%.2f%%', padding=3, fontsize=8)
    axes[0].set(ylabel='Coverage with positive evidence (%)', title='Calibration coverage')
    axes[0].set_ylim(0, max(coverage) * 125 + 1)
    polynomial = diagnostics.get('polynomial_dose_by_coordinate', {})
    if polynomial:
        coordinates = sorted(map(int, polynomial))
        values = [100 * polynomial[str(coordinate)] for coordinate in coordinates]
        axes[1].plot(coordinates, values, 'o-', color=COLORS['blue'], label='Polynomial probe dose')
    operator_dose = diagnostics.get('relative_operator_dose')
    if operator_dose is not None:
        axes[1].axhline(100 * operator_dose, color=COLORS['orange'], linestyle='--', label='Operator Frobenius dose')
    axes[1].set(xlabel='Embedding coordinate', ylabel='Relative dose (%)', title='Operator / polynomial\nprobe dose')
    axes[1].legend(frameon=False)
    levels, values = diagnostics.get('evidence_quantile_levels'), diagnostics.get('evidence_score_quantiles')
    if levels is not None and values is not None and len(levels) == len(values):
        axes[2].plot(np.asarray(levels) * 100, values, 'o-', color=COLORS['green'])
        pd.DataFrame({'quantile': levels, 'evidence_score': values}).to_csv(
            RUN_ROOT / 'tables' / f'{STEM}_evidence_quantiles.csv', index=False)
    else:
        axes[2].text(.5, .5, 'Score quantiles not measured', ha='center', va='center', transform=axes[2].transAxes)
    axes[2].set(xlabel='Quantile (%)', ylabel='Excess-evidence score', ylim=(0, 1), title='Evidence-score quantile profile')
    for ax in axes:
        ax.grid(axis='y', alpha=.2)
    write_json(RUN_ROOT/'tables'/f'{STEM}_calibration_figure_data.json',{
       'coverage_fraction':coverage,'operator_relative_dose':operator_dose,'polynomial_dose_by_coordinate':polynomial,
       'quantile_levels':levels,'score_quantiles':values,'reference':'uncalibrated semantic-only eta0',
       'probe_scope':diagnostics.get('polynomial_probe_scope')})
    fig.suptitle(f'ECLIPSE w/o CSE | {DATASET_TITLES[key]} | seed {SEED}', fontsize=11)
    save_figure(fig, f'{STEM}_calibration_diagnostics',
                'w/o CSE: PEEC remains active; reference is the uncalibrated semantic-only eta0 operator. '
                'Polynomial dose uses fixed Rademacher probes, not actual Adam directions. '
                'The score panel displays measured quantiles, not a reconstructed histogram.', [artifact / 'manifest.json'])


## 9. TRAIN objective and VALID ranking curves


In [ ]:
plot_learning_curves('electronics')


## 10. Training allocator memory and epoch time


In [ ]:
plot_training_memory('electronics')


## 11. Calibration diagnostics — semantic-only reference


In [ ]:
plot_calibration_diagnostics('electronics')


## 12. Selected TEST and resource summary — one measured run only
No historical seeds/Full metrics are injected. Coach.fit comes from native stdout; missing log timing is NOT_MEASURED, not zero. Subprocess wall and summed training epoch time remain separate.


In [ ]:
summary_rows = []
for key in SELECTED_DATASETS:
    if completed_artifact(key) is None:
        summary_rows.append({'method': 'ECLIPSE', 'dataset': key, 'seed': SEED, 'status': 'NOT_MEASURED'})
        continue
    summary = export_run_tables(key)
    names = ['method', 'arm', 'dataset', 'seed', 'selected_epoch', *PRIMARY_METRICS,
             'preprocessing_seconds', 'fit_seconds', 'train_epoch_seconds_sum',
             'peak_training_allocated_mib', 'peak_training_reserved_mib',
             'checkpoint_path', 'commit', 'graph_fingerprint']
    summary_rows.append({name: summary.get(name) for name in names} | {'status': 'COMPLETED'})
SUMMARY = pd.DataFrame(summary_rows)
for column in ['selected_epoch', *PRIMARY_METRICS, 'preprocessing_seconds', 'fit_seconds',
               'peak_training_allocated_mib', 'peak_training_reserved_mib']:
    if column not in SUMMARY:
        SUMMARY[column] = np.nan
SUMMARY.to_csv(RUN_ROOT / 'tables' / f'{STEM}_selected_test_overview.csv', index=False, na_rep='NOT_MEASURED')
display(SUMMARY[['dataset', 'seed', 'status', 'selected_epoch', *PRIMARY_METRICS]].fillna('NOT_MEASURED'))
measured = SUMMARY.loc[SUMMARY['status'] == 'COMPLETED'].copy()
if len(measured):
    labels = [DATASET_TITLES[key].replace('Amazon ', '') for key in measured['dataset']]
    x = np.arange(len(measured))
    fig, axes = plt.subplots(1, 3, figsize=(7.2, 3.0), constrained_layout=True)
    if measured['fit_seconds'].notna().any():
        axes[0].bar(x, pd.to_numeric(measured['fit_seconds'],errors='coerce') / 3600, color=COLORS['blue'])
    else:
        axes[0].text(.5,.5,'NOT_MEASURED',ha='center',va='center',transform=axes[0].transAxes)
    axes[0].set(ylabel='Coach.fit wall time (h)', title='Training + native evaluation / I/O')
    axes[1].bar(x, measured['preprocessing_seconds'], color=COLORS['green'])
    axes[1].set(ylabel='Preprocessing wall time (s)', title='Total model preprocessing')
    axes[2].bar(x - .18, measured['peak_training_allocated_mib'], width=.36, color=COLORS['blue'], label='Allocated')
    axes[2].bar(x + .18, measured['peak_training_reserved_mib'], width=.36, color=COLORS['orange'], label='Reserved')
    axes[2].set(ylabel='Training allocator peak (MiB)', title='Training-epoch CUDA memory')
    axes[2].legend(frameon=False)
    for ax in axes:
        ax.set_xticks(x, labels)
        ax.grid(axis='y', alpha=.2)
    fig.suptitle(f'ECLIPSE w/o CSE resource profile | seed {SEED}', fontsize=11)
    save_figure(fig, f'{STEM}_resource_profile',
                'ECLIPSE w/o CSE resource measurements, not a comparison against another model. '
                'Coach.fit includes native evaluation and I/O; preprocessing is separate. '
                'Memory excludes evaluation peaks. Cache state and runtime remain in manifests.',
                [RUN_ROOT / 'tables' / f'{STEM}_selected_test_overview.csv'])

resource_fields=['preprocessing_seconds','fit_seconds','train_epoch_seconds_sum','subprocess_wall_seconds','peak_training_allocated_mib','peak_training_reserved_mib','memory_scope','preprocessing_cache_hit']
pd.DataFrame([{k:RESULTS['electronics'].get(k) for k in resource_fields}]).to_csv(RUN_ROOT/'tables'/f'{STEM}_resource_summary.csv',index=False,na_rep='NOT_MEASURED')


## 13. Complete archive — native files + external log + source/config + figures/tables
ZIP is outside the run folder and uses a unique session ID. The inventory records bytes and SHA256; an external sidecar records archive bytes/SHA256. Read-only recovery copies existing artifacts into a new export folder without changing the original. Native checkpoint bytes are packaged but never loaded by reporting cells.

A nonrecursive source template includes every logical cell; it excludes only its own embedding assignment and runtime outputs. For the exact submitted Kaggle notebook, also set NOTEBOOK_SOURCE_PATH or download the executed notebook separately. Human cross-read remains pending.


In [ ]:
import zipfile
write_json(RUN_ROOT/'provenance'/'notebook_source_template.ipynb',NOTEBOOK_SOURCE_TEMPLATE)
script='\n\n'.join('# CELL '+str(i)+'\n'+''.join(c['source']) for i,c in enumerate(NOTEBOOK_SOURCE_TEMPLATE['cells']) if c['cell_type']=='code')
(RUN_ROOT/'helpers'/'orchestration_source.py').write_text(script,encoding='utf-8')
if NOTEBOOK_SOURCE_PATH:
    original=Path(NOTEBOOK_SOURCE_PATH).expanduser().resolve()
    if not original.is_file():raise FileNotFoundError(original)
    shutil.copy2(original,RUN_ROOT/'provenance'/'submitted_notebook.ipynb')
    write_json(RUN_ROOT/'provenance'/'submitted_notebook_receipt.json',{'path':str(original),'sha256':file_sha256(original)})
if RUN_MODE=='report_only':
    native=ARTIFACT_PATHS['electronics'];copied=RUN_ROOT/'artifacts'/STEM
    shutil.copytree(native,copied,dirs_exist_ok=False)
    log=LOG_PATHS['electronics']
    if log and log.is_file():shutil.copy2(log,RUN_ROOT/'logs'/f'{STEM}.log')
    m=json.loads((native/'manifest.json').read_text(encoding='utf-8'))
    write_json(RUN_ROOT/'provenance'/'report_only_origin.json',{'artifact':str(native),'copied_to':str(copied),
      'stdout_log':str(log) if log else None,'raw_artifacts_modified':False,'recovered_from_native_manifest':True,
      'original_yaml_available':False,'original_dirty_diff_available':False,'note':'Do not invent unavailable historical provenance.'})
    write_json(RUN_ROOT/'configs'/f'{STEM}_recorded_native_config.json',{'v8_config':m.get('v8_config'),'icaart_config':m.get('icaart_config'),
       'checkpoint_config':m.get('checkpoint_contract',{}).get('config'),'argv':m.get('argv')})
    write_json(RUN_ROOT/'provenance'/f'{STEM}_recorded_native_environment.json',{k:m.get(k) for k in ('python','torch','freerec','gpu','commit','precision')})
for kind in ('learning_curves','training_memory','calibration_diagnostics'):
    for extension in ('pdf','png'):
        path=RUN_ROOT/'figures'/f'{STEM}_{kind}.{extension}'
        if not path.is_file():raise FileNotFoundError(f'Missing required figure {path}; inspect actual diagnostics, do not fabricate values.')
entries=json.loads((RUN_ROOT/'figures'/'figure_manifest.json').read_text(encoding='utf-8'))
caption_text='# ECLIPSE w/o CSE figure captions\n\n'
for entry in entries:
    caption_text+='## '+entry['stem']+'\n\n'+entry['caption']+'\n\nSources:\n'+'\n'.join('- '+p for p in entry['sources'])+'\n\n'
(RUN_ROOT/'figures'/'FIGURE_CAPTIONS.md').write_text(caption_text,encoding='utf-8')
run=json.loads((RUN_ROOT/'run_manifest.json').read_text(encoding='utf-8'))
run.update(status='COMPLETED_PENDING_HUMAN_REVIEW',selected_epoch=RESULTS['electronics']['selected_epoch'],actual_n=1,
      comparability=RESULTS['electronics']['comparability']['status'],reviewed_by=None,matched_full_comparison='NOT_COMPUTED')
write_json(RUN_ROOT/'run_manifest.json',run)
(RUN_ROOT/'README.md').write_text(f"""# ECLIPSE w/o CSE | Electronics | seed {SEED}
Mode: {RUN_MODE}; unique attempt/export: {SESSION_ID}
Protocol: 64D, 500 epochs; eta0, PEEC delta1, LC .01. Native VALID-selected TEST only.
Pinned algorithm commit: {RUNTIME_COMMIT}; historical source differences require separate review.
Primary table: tables/{STEM}_selected_test.csv (canonical uppercase metrics).
Native receipt/checkpoint/telemetry: artifacts/{STEM}/
External stdout: logs/; configs/ and provenance/ retain configuration/source/environment evidence.
Figures: PDF vector + PNG300dpi, captions and source CSV/JSON. One measured run only; no seed SD/p-value.
Timing: preprocessing and Coach.fit separate; subprocess wall is not Coach.fit; missing data NOT_MEASURED.
Memory: child training-epoch CUDA allocator, excludes evaluation/driver/other GPU processes.
Source template includes logical cells without self-embedding. Exact submitted notebook copied only if supplied.
Human review remains pending. Reporting does not load checkpoint tensors or reproduce model metrics.
""",encoding='utf-8')
inventory=[]
for path in sorted(RUN_ROOT.rglob('*')):
    if path.is_file() and path.name!='file_inventory.json':
        inventory.append({'path':path.relative_to(RUN_ROOT).as_posix(),'bytes':path.stat().st_size,'sha256':file_sha256(path)})
write_json(RUN_ROOT/'file_inventory.json',inventory)
archive=RUN_ROOT.parent/f'{RUN_ROOT.name}.zip'
with zipfile.ZipFile(archive,'x',compression=zipfile.ZIP_DEFLATED,compresslevel=1) as z:
    for path in sorted(RUN_ROOT.rglob('*')):
        if path.is_file():z.write(path,arcname=str(Path(RUN_ROOT.name)/path.relative_to(RUN_ROOT)))
with zipfile.ZipFile(archive) as z:
    if z.testzip() is not None:raise RuntimeError('Archive CRC validation failed')
receipt={'archive':str(archive),'bytes':archive.stat().st_size,'sha256':file_sha256(archive),'file_count':len(inventory)+1,
         'status':'PACKAGED_PENDING_HUMAN_REVIEW','run_root':str(RUN_ROOT)}
write_json(archive.with_suffix('.archive_receipt.json'),receipt)
print(json.dumps(receipt,indent=2))
try:
    from IPython.display import FileLink
    display(FileLink(str(archive)));display(FileLink(str(archive.with_suffix('.archive_receipt.json'))))
except ImportError:pass
